In [ ]:
# 0) Environment + reusable runner (run first)
from pathlib import Path
import os, sys, json, subprocess, random, shutil
from datetime import datetime

_cwd = Path.cwd().resolve()
_candidates = [
    _cwd,
    _cwd.parent,
    _cwd / 'backend',
    _cwd.parent / 'backend',
    Path('/content/CausalX-Project/backend'),
    Path('/content/drive/MyDrive/CausalX-Project/backend'),
]
PROJECT_ROOT = next((p for p in _candidates if (p / 'src').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate backend root with src/ directory.')

def _load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = raw.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('export '):
            line = line[len('export '):].strip()
        if '=' not in line:
            continue
        k, v = line.split('=', 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k and k not in os.environ:
            os.environ[k] = os.path.expandvars(os.path.expanduser(v))

_load_env_file(PROJECT_ROOT / 'configs' / 'dataset_paths.env')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

VENV_PY = PROJECT_ROOT / '.venv' / 'bin' / 'python'
PY_BIN = str(VENV_PY if VENV_PY.exists() else Path(sys.executable))

os.environ['PYTHONPATH'] = str(PROJECT_ROOT)
os.environ.setdefault('CFN_USE_EMBEDDINGS', 'true')
os.environ.setdefault('CFN_W2V2_MODEL', 'WAV2VEC2_BASE')
os.environ.setdefault('CFN_EMB_MODEL_PATH', str(PROJECT_ROOT / 'models' / 'cfn_emb.pth'))
os.environ.setdefault('CFN_VISUAL_TCN_PATH', str(PROJECT_ROOT / 'models' / 'visual_tcn.pth'))
os.environ.setdefault('MEDIAPIPE_DISABLE_GPU', '1')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '-1')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = PROJECT_ROOT / 'models' / 'experiment_logs' / f'pipeline_min_{RUN_ID}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, name, extra_env=None, check=True):
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    print('$ ' + ' '.join(cmd))
    res = subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=env, text=True, capture_output=True)
    (RUN_DIR / f'{name}.stdout.log').write_text(res.stdout or '')
    (RUN_DIR / f'{name}.stderr.log').write_text(res.stderr or '')
    (RUN_DIR / f'{name}.meta.json').write_text(json.dumps({'cmd': cmd, 'returncode': res.returncode}, indent=2))
    if res.stdout:
        print(res.stdout[-4000:])
    if res.stderr:
        print('--- stderr ---')
        print(res.stderr[-4000:])
    if check and res.returncode != 0:
        raise RuntimeError(f'{name} failed with code {res.returncode}')
    return res

def snapshot_model(tag):
    src_model = PROJECT_ROOT / 'models' / 'cfn_emb.pth'
    src_scaler = PROJECT_ROOT / 'models' / 'cfn_scaler.pkl'
    if src_model.exists():
        shutil.copy2(src_model, RUN_DIR / f'{tag}.cfn_emb.pth')
    if src_scaler.exists():
        shutil.copy2(src_scaler, RUN_DIR / f'{tag}.cfn_scaler.pkl')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PY_BIN:', PY_BIN)
print('RUN_DIR:', RUN_DIR)


In [ ]:
# 1) Base retrain (all sources)
run_cmd([
    PY_BIN, '-m', 'src.training.train_cfn',
    '--data', 'data/processed/causal_multimodal_dataset.csv',
    '--train-source', 'all',
    '--use-embeddings', '--use-scaler',
    '--group-balance', '--use-weighted-sampler',
    '--loss', 'focal', '--focal-alpha', '0.75', '--focal-gamma', '2.0',
    '--causal-weight', '0.15',
    '--scheduler', 'cosine',
    '--epochs', '30', '--patience', '8', '--batch-size', '128',
    '--lr', '3e-4', '--weight-decay', '1e-4',
    '--selection-metric', 'hybrid_robust',
    '--selection-threshold', '0.5',
    '--min-domain-spec', '0.30',
    '--min-domain-rec', '0.50',
], '01_train_base')
snapshot_model('01_train_base')


In [ ]:
# 7) Apply selected env + save final report
for k, v in SELECTED_PAYLOAD['recommend_env'].items():
    os.environ[k] = v

best = SELECTED_PAYLOAD['best']
overall = best['metrics']['overall']
per_ds = best['metrics']['per_dataset']

print('Best config:')
print('  PROB=', best['prob'], 'RATIO=', best['ratio'], 'CAUSAL=', best['causal'], 'REQUIRE_FLAG=', best['require_flag'])
print('Overall:')
print('  Acc={acc:.3f} BalAcc={bal_acc:.3f} F1={f1:.3f} Rec={rec:.3f} Spec={spec:.3f}'.format(**overall))
print('Per dataset:')
for ds, m in per_ds.items():
    print(f"  [{ds}] Acc={m['acc']:.3f} BalAcc={m['bal_acc']:.3f} F1={m['f1']:.3f} Rec={m['rec']:.3f} Spec={m['spec']:.3f}")

print('Applied env:')
for k in ['CFN_PROB_THRESH', 'CFN_RATIO_THRESH', 'CFN_CAUSAL_THRESH', 'CFN_REQUIRE_FLAG']:
    print(k, '=', os.environ.get(k))

report = {
    'run_id': RUN_ID,
    'run_dir': str(RUN_DIR),
    'selected_metrics': overall,
    'per_dataset': per_ds,
    'recommend_env': SELECTED_PAYLOAD['recommend_env'],
}
report_path = RUN_DIR / 'final_report.json'
report_path.write_text(json.dumps(report, indent=2))
print('Saved report:', report_path)


## 20) Preprocessing -> Full Pipeline (Audio Trim + Feature Refresh + Sweeps)

Use this section when you want to rerun the pipeline from preprocessing onward, in one place:
1. Trim first 100ms of FakeAVCeleb audio (bias mitigation).
2. Re-extract features into `data/processed/causal_multimodal_dataset.csv`.
3. Rebuild full-train + balanced holdout splits.
4. Run baseline and/or objective sweeps.

All heavy steps are toggle-gated (`False` by default).


In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd

# ---- root / python resolution ----
if 'BACKEND_ROOT' not in globals() or BACKEND_ROOT is None:
    _candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().resolve() / 'backend',
        Path('/content/CausalX-Project/backend'),
    ]
    BACKEND_ROOT = next((c for c in _candidates if (c / 'src').exists()), None)

if BACKEND_ROOT is None:
    raise RuntimeError('Cannot find backend root with src/ directory.')

PYTHON = BACKEND_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON.exists():
    PYTHON = Path('python3')

# ---- key paths ----
FAKEAV_ROOT = BACKEND_ROOT / 'data/raw/fakeavceleb'
PROCESSED_CSV = BACKEND_ROOT / 'data/processed/causal_multimodal_dataset.csv'
TRIM_MANIFEST = BACKEND_ROOT / 'data/processed/fakeav_audio_trim_manifest.csv'

# ---- split config ----
TRAIN_MAX_FAKE_MULTIPLIER = None  # e.g., None (default), 32.0, 8.0
if TRAIN_MAX_FAKE_MULTIPLIER is None:
    SPLIT_TAG = 'fulltrain'
else:
    SPLIT_TAG = f"cap{int(round(float(TRAIN_MAX_FAKE_MULTIPLIER)))}"

FULL_TRAIN = BACKEND_ROOT / f'data/processed/causal_multimodal_dataset_fakeav_{SPLIT_TAG}_train.csv'
FULL_VAL = BACKEND_ROOT / f'data/processed/causal_multimodal_dataset_fakeav_{SPLIT_TAG}_val.csv'
FULL_TEST = BACKEND_ROOT / f'data/processed/causal_multimodal_dataset_fakeav_{SPLIT_TAG}_test.csv'
FULL_META = BACKEND_ROOT / f'data/processed/causal_multimodal_dataset_fakeav_{SPLIT_TAG}_meta.json'

# ---- toggles ----
RUN_AUDIO_TRIM = False
RUN_FEATURE_EXTRACTION = False
RUN_BUILD_FULLTRAIN_SPLITS = False
RUN_FULLTRAIN_BASE_SWEEP = False
RUN_FULLTRAIN_OBJECTIVE_SWEEP = False

# ---- params ----
TRIM_SECONDS = 0.10
FULLTRAIN_BASE_PREFIX = f'fakeav_auc_sweep_{SPLIT_TAG}_base_thrcal'
FULLTRAIN_OBJ_PREFIX = f'fakeav_auc_sweep_{SPLIT_TAG}_obj_thrcal'

print('BACKEND_ROOT:', BACKEND_ROOT)
print('PYTHON:', PYTHON)
print('SPLIT_TAG:', SPLIT_TAG)
print('TRAIN_MAX_FAKE_MULTIPLIER:', TRAIN_MAX_FAKE_MULTIPLIER)
print('Toggles:', {
    'RUN_AUDIO_TRIM': RUN_AUDIO_TRIM,
    'RUN_FEATURE_EXTRACTION': RUN_FEATURE_EXTRACTION,
    'RUN_BUILD_FULLTRAIN_SPLITS': RUN_BUILD_FULLTRAIN_SPLITS,
    'RUN_FULLTRAIN_BASE_SWEEP': RUN_FULLTRAIN_BASE_SWEEP,
    'RUN_FULLTRAIN_OBJECTIVE_SWEEP': RUN_FULLTRAIN_OBJECTIVE_SWEEP,
})


In [ ]:
# Step 1: FakeAVCeleb audio-bias mitigation (trim first 100ms)
from pathlib import Path
if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'scripts' / 'trim_fakeav_audio_head.py').exists():
    _cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in _cands if (p / 'scripts' / 'trim_fakeav_audio_head.py').exists()), _cands[-1])
TRIM_MANIFEST = BACKEND_ROOT / 'data/processed/fakeav_audio_trim_manifest.csv'
if RUN_AUDIO_TRIM:
    import sys
    if 'PYTHON' in globals():
        PYTHON = Path(PYTHON)
    if 'PYTHON' not in globals() or not PYTHON.exists():
        py_cand = BACKEND_ROOT / '.venv' / 'bin' / 'python'
        PYTHON = py_cand if py_cand.exists() else Path(sys.executable)
    script_path = BACKEND_ROOT / 'scripts/trim_fakeav_audio_head.py'
    cmd = [
        str(PYTHON), str(script_path),
        '--input-root', 'data/raw/fakeavceleb',
        '--in-place',
        '--trim-seconds', f'{TRIM_SECONDS:.2f}',
        '--manifest-csv', str(TRIM_MANIFEST.relative_to(BACKEND_ROOT)),
    ]
    print('CMD:', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'trim_fakeav_audio_head.py failed: code={res.returncode}')
else:
    print('RUN_AUDIO_TRIM=False (skipping audio trim).')

if TRIM_MANIFEST.exists():
    man = pd.read_csv(TRIM_MANIFEST)
    print('Latest trim status counts:')
    print(man['status'].value_counts(dropna=False).to_string())
    print('Manifest:', TRIM_MANIFEST)


In [ ]:
# Step 2: Feature extraction refresh
if RUN_FEATURE_EXTRACTION:
    cmd = [str(PYTHON), '-m', 'src.preprocessing.batch_feature_extractor']
    print('CMD:', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'batch_feature_extractor failed: code={res.returncode}')
else:
    print('RUN_FEATURE_EXTRACTION=False (skipping extraction).')

if PROCESSED_CSV.exists():
    df = pd.read_csv(PROCESSED_CSV)
    print('Processed CSV:', PROCESSED_CSV)
    print('Rows:', len(df), 'Columns:', len(df.columns))
    if 'dataset' in df.columns and 'label' in df.columns:
        print('Dataset counts:')
        print(df['dataset'].astype(str).str.lower().value_counts().to_string())
        print('Label counts:')
        print(df['label'].astype(int).value_counts().sort_index().to_string())


In [ ]:
# Step 3: Build full-train + balanced holdout splits
if RUN_BUILD_FULLTRAIN_SPLITS:
    cmd = [
        str(PYTHON), 'scripts/build_fakeav_holdout_balanced_splits.py',
        '--processed-csv', str(PROCESSED_CSV.relative_to(BACKEND_ROOT)),
        '--seed', '42',
        '--val-size', '0.15',
        '--test-size', '0.15',
        '--holdout-fake-strategy', 'scenario_balanced',
        '--out-train-csv', str(FULL_TRAIN.relative_to(BACKEND_ROOT)),
        '--out-val-csv', str(FULL_VAL.relative_to(BACKEND_ROOT)),
        '--out-test-csv', str(FULL_TEST.relative_to(BACKEND_ROOT)),
        '--out-meta-json', str(FULL_META.relative_to(BACKEND_ROOT)),
    ]
    if TRAIN_MAX_FAKE_MULTIPLIER is not None:
        cmd.extend(['--train-max-fake-multiplier', str(float(TRAIN_MAX_FAKE_MULTIPLIER))])
    print('CMD:', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'build_fakeav_holdout_balanced_splits.py failed: code={res.returncode}')
else:
    print('RUN_BUILD_FULLTRAIN_SPLITS=False (skipping split build).')

if FULL_META.exists():
    meta = json.loads(FULL_META.read_text())
    print('Split summary:')
    print(json.dumps(meta.get('splits', {}), indent=2))
    if 'train_fakeav_label_counts_before_cap' in meta:
        print('Cap info:')
        print({
            'train_max_fake_multiplier': meta.get('train_max_fake_multiplier'),
            'before': meta.get('train_fakeav_label_counts_before_cap'),
            'after': meta.get('train_fakeav_label_counts_after_cap'),
            'rows_removed': meta.get('train_fakeav_rows_removed_by_cap'),
        })


In [ ]:
# Step 4A: Full-train baseline sweep (no multitask/ranking)
if RUN_FULLTRAIN_BASE_SWEEP:
    cmd = [
        str(PYTHON), 'scripts/sweep_fakeav_auc.py',
        '--train-csv', str(FULL_TRAIN.relative_to(BACKEND_ROOT)),
        '--val-csv', str(FULL_VAL.relative_to(BACKEND_ROOT)),
        '--test-csv', str(FULL_TEST.relative_to(BACKEND_ROOT)),
        '--feature-profile', 'extended',
        '--epochs', '35',
        '--patience', '8',
        '--max-runs', '10',
        '--train-source', 'fakeavceleb',
        '--train-use-weighted-sampler',
        '--train-weight-application', 'auto',
        '--eval-threshold-source', 'val_target',
        '--eval-threshold-priority', 'balanced_acc',
        '--run-prefix', FULLTRAIN_BASE_PREFIX,
    ]
    print('CMD:', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'fulltrain baseline sweep failed: code={res.returncode}')
else:
    print('RUN_FULLTRAIN_BASE_SWEEP=False (skipping baseline sweep).')


In [ ]:
# Step 4B: Full-train objective sweep (multitask + ranking)
if RUN_FULLTRAIN_OBJECTIVE_SWEEP:
    cmd = [
        str(PYTHON), 'scripts/sweep_fakeav_auc.py',
        '--train-csv', str(FULL_TRAIN.relative_to(BACKEND_ROOT)),
        '--val-csv', str(FULL_VAL.relative_to(BACKEND_ROOT)),
        '--test-csv', str(FULL_TEST.relative_to(BACKEND_ROOT)),
        '--feature-profile', 'extended',
        '--epochs', '35',
        '--patience', '8',
        '--max-runs', '10',
        '--train-source', 'fakeavceleb',
        '--train-use-weighted-sampler',
        '--train-weight-application', 'auto',
        '--enable-multitask',
        '--multitask-weight', '0.20',
        '--ranking-loss-weight', '0.10',
        '--ranking-margin', '0.20',
        '--ranking-max-pairs', '1024',
        '--eval-threshold-source', 'val_target',
        '--eval-threshold-priority', 'balanced_acc',
        '--run-prefix', FULLTRAIN_OBJ_PREFIX,
    ]
    print('CMD:', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'fulltrain objective sweep failed: code={res.returncode}')
else:
    print('RUN_FULLTRAIN_OBJECTIVE_SWEEP=False (skipping objective sweep).')


In [ ]:
# Step 5: Quick leaderboard check for this split tag
logs_root = BACKEND_ROOT / 'models/experiment_logs'
patterns = [f'{FULLTRAIN_BASE_PREFIX}*', f'{FULLTRAIN_OBJ_PREFIX}*']

for pat in patterns:
    dirs = sorted([p for p in logs_root.glob(pat) if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
    if not dirs:
        print(f'No directories for pattern: {pat}')
        continue
    latest = dirs[0]
    lb = latest / 'leaderboard_sorted.csv'
    print('\nPattern:', pat)
    print('Latest :', latest)
    if lb.exists():
        df = pd.read_csv(lb)
        if len(df):
            cols = ['run_tag', 'target_gap_total', 'test_auc', 'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_thr']
            cols = [c for c in cols if c in df.columns]
            display(df[cols].head(5))
        else:
            print('Leaderboard exists but empty.')
    else:
        print('Missing leaderboard_sorted.csv')


## 14) MRDF-Balanced 5-Fold CV Protocol

This section follows the requested protocol:
- MRDF-style balanced 5-fold cross-validation split
- WeightedRandomSampler during training
- Focal Loss (or weighted BCE)
- Metrics: balanced accuracy, AUC, class-wise F1, per-manipulation AUC


In [ ]:
from pathlib import Path
import subprocess

# Self-healing bootstrap: this cell can run standalone.
if 'BACKEND_ROOT' not in globals() or BACKEND_ROOT is None:
    _candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().resolve() / 'backend',
        Path('/content/CausalX-Project/backend'),
    ]
    BACKEND_ROOT = next((c for c in _candidates if (c / 'src').exists()), None)

if BACKEND_ROOT is None:
    raise RuntimeError('Cannot find backend root with src/ directory.')

PYTHON = BACKEND_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON.exists():
    PYTHON = Path('python3')

PROCESSED_CSV = BACKEND_ROOT / 'data/processed/causal_multimodal_dataset.csv'
if not PROCESSED_CSV.exists():
    raise RuntimeError(f'Missing processed CSV: {PROCESSED_CSV}')

# Configure and run MRDF 5-fold CV
RUN_MRDF_5FOLD_CV = True
MRDF_RUN_PREFIX = 'fakeav_mrdf5cv'
MRDF_SEED = 42
MRDF_SEED_LIST = [42, 1337, 2026]  # leave as [] for single-seed mode
MRDF_SPLIT_SEED = 42
MRDF_ENSEMBLE_ENABLE = True
MRDF_ENSEMBLE_TOP_K = 3            # 0 => use all seeds
MRDF_ENSEMBLE_RANK_METRIC = 'val_bal_acc'
MRDF_ENSEMBLE_WEIGHTING = 'uniform'  # uniform|rank_metric

MRDF_N_SPLITS = 5
MRDF_MAX_FOLDS = None       # set e.g. 1 for smoke test
MRDF_VAL_SIZE = 0.15
MRDF_PER_SCENARIO = None    # None => max balanced subset

MRDF_LOSS = 'focal'         # 'focal' or 'bce'
MRDF_EPOCHS = 35
MRDF_PATIENCE = 8
MRDF_BATCH_SIZE = 128
MRDF_LR = 3e-4
MRDF_WEIGHT_DECAY = 1e-4
MRDF_CAUSAL_WEIGHT = 0.0
MRDF_FOCAL_ALPHA = 0.40
MRDF_FOCAL_GAMMA = 2.0
MRDF_WEIGHT_APPLICATION = 'both'  # required for weighted BCE path

# Objective upgrades
MRDF_ENABLE_MULTITASK = True
MRDF_MULTITASK_WEIGHT = 0.25
MRDF_RANKING_LOSS_WEIGHT = 0.20
MRDF_RANKING_MARGIN = 0.20
MRDF_RANKING_MAX_PAIRS = 1024

# Optional phase-1 scenario focus
MRDF_TRAIN_SCENARIO_FOCUS = 'none'            # none|video_only_fake|audio_only_fake|both_fake
MRDF_TRAIN_SCENARIO_FOCUS_WEIGHT = 1.0

# Optional global pretrain before fold fine-tuning
MRDF_PRETRAIN_ENABLE = True
MRDF_PRETRAIN_EPOCHS = 12
MRDF_PRETRAIN_PATIENCE = 4
MRDF_PRETRAIN_SCENARIO_FOCUS = 'none'
MRDF_PRETRAIN_SCENARIO_FOCUS_WEIGHT = 1.0

# Optional phase-2 hard-example retraining
MRDF_PHASE2_ENABLE = True
MRDF_PHASE2_HARDNEG_SOURCE = 'train'          # train|val
MRDF_PHASE2_ROUNDS = 2
MRDF_PHASE2_USE_HARD_POSITIVES = False
MRDF_PHASE2_HARDPOS_SOURCE = 'train'          # train|val
MRDF_HARDNEG_WEIGHT = 4.0
MRDF_HARDPOS_WEIGHT = 2.0
MRDF_HARDPOS_SCENARIO = 'audio_only_fake'     # none|video_only_fake|audio_only_fake|both_fake
MRDF_PHASE2_SCENARIO_FOCUS = 'none'  # inherit|none|video_only_fake|audio_only_fake|both_fake
MRDF_PHASE2_SCENARIO_FOCUS_WEIGHT = 1.0

cmd = [
    str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
    '--processed-csv', str(PROCESSED_CSV.relative_to(BACKEND_ROOT)),
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models',
    '--logs-dir', 'models/experiment_logs',
    '--run-prefix', MRDF_RUN_PREFIX,
    '--seed', str(int(MRDF_SEED)),
    '--split-seed', str(int(MRDF_SPLIT_SEED)),
    '--n-splits', str(int(MRDF_N_SPLITS)),
    '--val-size', str(float(MRDF_VAL_SIZE)),
    '--feature-profile', 'extended',
    '--epochs', str(int(MRDF_EPOCHS)),
    '--patience', str(int(MRDF_PATIENCE)),
    '--batch-size', str(int(MRDF_BATCH_SIZE)),
    '--lr', str(float(MRDF_LR)),
    '--weight-decay', str(float(MRDF_WEIGHT_DECAY)),
    '--causal-weight', str(float(MRDF_CAUSAL_WEIGHT)),
    '--loss', str(MRDF_LOSS),
    '--focal-alpha', str(float(MRDF_FOCAL_ALPHA)),
    '--focal-gamma', str(float(MRDF_FOCAL_GAMMA)),
    '--train-weight-application', str(MRDF_WEIGHT_APPLICATION),
    '--train-scenario-focus', str(MRDF_TRAIN_SCENARIO_FOCUS),
    '--train-scenario-focus-weight', str(float(MRDF_TRAIN_SCENARIO_FOCUS_WEIGHT)),
    '--multitask-weight', str(float(MRDF_MULTITASK_WEIGHT)),
    '--ranking-loss-weight', str(float(MRDF_RANKING_LOSS_WEIGHT)),
    '--ranking-margin', str(float(MRDF_RANKING_MARGIN)),
    '--ranking-max-pairs', str(int(MRDF_RANKING_MAX_PAIRS)),
    '--pretrain-epochs', str(int(MRDF_PRETRAIN_EPOCHS)),
    '--pretrain-patience', str(int(MRDF_PRETRAIN_PATIENCE)),
    '--pretrain-scenario-focus', str(MRDF_PRETRAIN_SCENARIO_FOCUS),
    '--pretrain-scenario-focus-weight', str(float(MRDF_PRETRAIN_SCENARIO_FOCUS_WEIGHT)),
    '--eval-threshold-source', 'val_target',
    '--eval-threshold-priority', 'balanced_acc',
    '--ensemble-top-k', str(int(MRDF_ENSEMBLE_TOP_K)),
    '--ensemble-rank-metric', str(MRDF_ENSEMBLE_RANK_METRIC),
    '--ensemble-weighting', str(MRDF_ENSEMBLE_WEIGHTING),
]

if MRDF_ENABLE_MULTITASK:
    cmd.append('--enable-multitask')
if MRDF_SEED_LIST:
    cmd.extend(['--seed-list', ','.join(str(int(s)) for s in MRDF_SEED_LIST)])
if MRDF_ENSEMBLE_ENABLE:
    cmd.append('--ensemble-enable')
if MRDF_PRETRAIN_ENABLE:
    cmd.append('--pretrain-enable')
if MRDF_PHASE2_ENABLE:
    cmd.append('--phase2-enable')
    cmd.extend(['--phase2-hardneg-source', str(MRDF_PHASE2_HARDNEG_SOURCE)])
    cmd.extend(['--phase2-rounds', str(int(MRDF_PHASE2_ROUNDS))])
    cmd.extend(['--phase2-hardpos-source', str(MRDF_PHASE2_HARDPOS_SOURCE)])
    cmd.extend(['--hard-negative-weight', str(float(MRDF_HARDNEG_WEIGHT))])
    cmd.extend(['--hard-positive-weight', str(float(MRDF_HARDPOS_WEIGHT))])
    cmd.extend(['--hardpos-scenario', str(MRDF_HARDPOS_SCENARIO)])
    cmd.extend(['--phase2-scenario-focus', str(MRDF_PHASE2_SCENARIO_FOCUS)])
    cmd.extend(['--phase2-scenario-focus-weight', str(float(MRDF_PHASE2_SCENARIO_FOCUS_WEIGHT))])
    if MRDF_PHASE2_USE_HARD_POSITIVES:
        cmd.append('--phase2-use-hard-positives')

if MRDF_PER_SCENARIO is not None:
    cmd.extend(['--per-scenario', str(int(MRDF_PER_SCENARIO))])
if MRDF_MAX_FOLDS is not None:
    cmd.extend(['--max-folds', str(int(MRDF_MAX_FOLDS))])

print('BACKEND_ROOT:', BACKEND_ROOT)
print('PYTHON:', PYTHON)
print('CMD:', ' '.join(cmd))
if RUN_MRDF_5FOLD_CV:
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'MRDF 5-fold CV failed: code={res.returncode}')
else:
    print('RUN_MRDF_5FOLD_CV=False (skipping execution). Set to True to run.')







In [ ]:
from pathlib import Path
import json
import pandas as pd

# Self-healing bootstrap: this cell can run standalone.
if 'BACKEND_ROOT' not in globals() or BACKEND_ROOT is None:
    _candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().resolve() / 'backend',
        Path('/content/CausalX-Project/backend'),
    ]
    BACKEND_ROOT = next((c for c in _candidates if (c / 'src').exists()), None)

if BACKEND_ROOT is None:
    raise RuntimeError('Cannot find backend root with src/ directory.')

if 'MRDF_RUN_PREFIX' not in globals():
    MRDF_RUN_PREFIX = 'fakeav_mrdf5cv'

# Load latest MRDF CV summary + fold metrics
logs_root = BACKEND_ROOT / 'models/experiment_logs'
cv_dirs = sorted(
    [p for p in logs_root.glob(f'{MRDF_RUN_PREFIX}_*') if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if not cv_dirs:
    print(f'No MRDF CV runs found for prefix: {MRDF_RUN_PREFIX}')
else:
    latest_cv = cv_dirs[0]
    fold_csv = latest_cv / 'fold_metrics.csv'
    summary_json = latest_cv / 'cv_summary.json'
    print('Latest MRDF CV run:', latest_cv)

    if fold_csv.exists():
        fold_df = pd.read_csv(fold_csv)
        show_cols = [
            'fold', 'phase_decision', 'phase2_applied',
            'phase2_hard_negative_rows', 'phase2_hard_positive_rows',
            'loss', 'train_weighted_sampler', 'train_weight_application',
            'test_acc', 'test_prec', 'test_rec', 'test_f1',
            'test_bal_acc', 'test_auc', 'test_f1_real', 'test_f1_fake',
            'test_auc_manip_audio_only', 'test_auc_manip_video_only', 'test_auc_manip_both_fake',
            'error'
        ]
        show_cols = [c for c in show_cols if c in fold_df.columns]
        display(fold_df[show_cols])
    else:
        print('Missing fold_metrics.csv')

    if summary_json.exists():
        summary = json.loads(summary_json.read_text())
        print('cv_summary.json:')
        print(json.dumps(summary, indent=2))
    else:
        print('Missing cv_summary.json')



## 15) 98%-Accuracy Target Push (Documented Changes)

This section documents all new changes made during the latest improvement cycle:
- Added multi-seed + built-in ensemble options in `scripts/run_fakeav_mrdf_5fold_cv.py`
- Added feature-dimension compatibility alignment for old/new scaler/model artifacts in eval/sweep scripts
- Added engineered video-artifact features to `src/cvi/feature_schema.py`
- Executed step-wise experiments and exported comparison CSVs against updated targets (Accuracy=0.98, Precision=0.87, Recall=0.88, F1=0.88, AUC=0.91).


In [ ]:
from pathlib import Path
import json
import pandas as pd

if 'BACKEND_ROOT' not in globals() or BACKEND_ROOT is None:
    _candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().resolve() / 'backend',
        Path('/content/CausalX-Project/backend'),
    ]
    BACKEND_ROOT = next((c for c in _candidates if (c / 'src').exists()), None)
if BACKEND_ROOT is None:
    raise RuntimeError('Cannot find backend root with src/ directory.')

PYTHON = BACKEND_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON.exists():
    PYTHON = Path('python3')

UPDATED_TARGETS = {
    'test_acc': 0.98,
    'test_prec': 0.87,
    'test_rec': 0.88,
    'test_f1': 0.88,
    'test_auc': 0.91,
}

print('BACKEND_ROOT:', BACKEND_ROOT)
print('PYTHON:', PYTHON)
print('UPDATED_TARGETS:', UPDATED_TARGETS)


### 15.1 Step-1 AUC-Recovery Experiment (Video-Only Hard-Positive Focus)

This run increases sensitivity to video-only failures in phase-2 by enabling hard positives and focusing on `video_only_fake` in both mining and phase-2 weighting.


In [ ]:
import subprocess

RUN_STEP1_VIDEO_ONLY = False

cmd = [
    str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
    '--processed-csv', 'data/processed/causal_multimodal_dataset.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models', '--logs-dir', 'models/experiment_logs',
    '--run-prefix', 'fakeav_mrdf5cv_step1_videoonly_focus',
    '--seed', '1337', '--split-seed', '42',
    '--n-splits', '5', '--val-size', '0.15',
    '--feature-profile', 'extended',
    '--epochs', '35', '--patience', '8', '--batch-size', '128',
    '--lr', '3e-4', '--weight-decay', '1e-4', '--causal-weight', '0.0',
    '--loss', 'focal', '--focal-alpha', '0.40', '--focal-gamma', '2.0',
    '--train-weight-application', 'both',
    '--enable-multitask', '--multitask-weight', '0.25',
    '--ranking-loss-weight', '0.20', '--ranking-margin', '0.20', '--ranking-max-pairs', '1024',
    '--pretrain-enable', '--pretrain-epochs', '12', '--pretrain-patience', '4',
    '--phase2-enable', '--phase2-rounds', '2', '--phase2-hardneg-source', 'train',
    '--phase2-use-hard-positives', '--phase2-hardpos-source', 'train',
    '--hard-negative-weight', '2.5', '--hard-positive-weight', '2.0',
    '--hardpos-scenario', 'video_only_fake',
    '--phase2-scenario-focus', 'video_only_fake', '--phase2-scenario-focus-weight', '1.75',
    '--eval-threshold-source', 'val_target', '--eval-threshold-priority', 'balanced_acc',
]
print('CMD:', ' '.join(cmd))
if RUN_STEP1_VIDEO_ONLY:
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'Step-1 video-only focus run failed: code={res.returncode}')
else:
    print('RUN_STEP1_VIDEO_ONLY=False (skipping execution).')


### 15.2 Step-2 Objective Sweep (Rank-Loss x Focal-Alpha)

Four combinations around the current best configuration are used to probe AUC/accuracy tradeoffs.


In [ ]:
import subprocess

RUN_STEP2_SWEEP = False
STEP2_CONFIGS = [
    ('fakeav_mrdf5cv_step2_r020_a035', '0.20', '0.35'),
    ('fakeav_mrdf5cv_step2_r020_a040', '0.20', '0.40'),
    ('fakeav_mrdf5cv_step2_r030_a035', '0.30', '0.35'),
    ('fakeav_mrdf5cv_step2_r030_a040', '0.30', '0.40'),
]

for run_prefix, rank_w, focal_alpha in STEP2_CONFIGS:
    cmd = [
        str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
        '--processed-csv', 'data/processed/causal_multimodal_dataset.csv',
        '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
        '--models-dir', 'models', '--logs-dir', 'models/experiment_logs',
        '--run-prefix', run_prefix, '--seed', '1337', '--split-seed', '42',
        '--n-splits', '5', '--val-size', '0.15', '--feature-profile', 'extended',
        '--epochs', '35', '--patience', '8', '--batch-size', '128',
        '--lr', '3e-4', '--weight-decay', '1e-4', '--causal-weight', '0.0',
        '--loss', 'focal', '--focal-alpha', focal_alpha, '--focal-gamma', '2.0',
        '--train-weight-application', 'both',
        '--enable-multitask', '--multitask-weight', '0.25',
        '--ranking-loss-weight', rank_w, '--ranking-margin', '0.20', '--ranking-max-pairs', '1024',
        '--pretrain-enable', '--pretrain-epochs', '12', '--pretrain-patience', '4',
        '--phase2-enable', '--phase2-rounds', '2', '--phase2-hardneg-source', 'train',
        '--phase2-hardpos-source', 'train',
        '--hard-negative-weight', '4.0', '--hard-positive-weight', '2.0',
        '--hardpos-scenario', 'audio_only_fake',
        '--phase2-scenario-focus', 'none', '--phase2-scenario-focus-weight', '1.0',
        '--eval-threshold-source', 'val_target', '--eval-threshold-priority', 'balanced_acc',
    ]
    print('\nCMD:', ' '.join(cmd))
    if RUN_STEP2_SWEEP:
        res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
        print(res.stdout[-4000:])
        if res.returncode != 0:
            print(res.stderr[-4000:])
            raise RuntimeError(f'Step-2 run failed for {run_prefix}: code={res.returncode}')

if not RUN_STEP2_SWEEP:
    print('RUN_STEP2_SWEEP=False (commands printed only).')


### 15.3 Step-3 Multi-Seed Ensemble (Top-2 by Val-AUC)

This run executes seeds `[42, 1337, 2026]` with fixed folds (`split_seed=42`) and ensemble selection by per-fold `val_auc`, `top_k=2`.


In [ ]:
import subprocess

RUN_STEP3_ENSEMBLE = False
cmd = [
    str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
    '--processed-csv', 'data/processed/causal_multimodal_dataset.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models', '--logs-dir', 'models/experiment_logs',
    '--run-prefix', 'fakeav_mrdf5cv_step3_top2valauc',
    '--seed', '42', '--seed-list', '42,1337,2026', '--split-seed', '42',
    '--n-splits', '5', '--val-size', '0.15', '--feature-profile', 'extended',
    '--epochs', '35', '--patience', '8', '--batch-size', '128',
    '--lr', '3e-4', '--weight-decay', '1e-4', '--causal-weight', '0.0',
    '--loss', 'focal', '--focal-alpha', '0.40', '--focal-gamma', '2.0',
    '--train-weight-application', 'both',
    '--enable-multitask', '--multitask-weight', '0.25',
    '--ranking-loss-weight', '0.20', '--ranking-margin', '0.20', '--ranking-max-pairs', '1024',
    '--pretrain-enable', '--pretrain-epochs', '12', '--pretrain-patience', '4',
    '--phase2-enable', '--phase2-rounds', '2', '--phase2-hardneg-source', 'train',
    '--phase2-hardpos-source', 'train', '--hard-negative-weight', '4.0', '--hard-positive-weight', '2.0',
    '--hardpos-scenario', 'audio_only_fake', '--phase2-scenario-focus', 'none', '--phase2-scenario-focus-weight', '1.0',
    '--eval-threshold-source', 'val_target', '--eval-threshold-priority', 'balanced_acc',
    '--ensemble-enable', '--ensemble-top-k', '2', '--ensemble-rank-metric', 'val_auc', '--ensemble-weighting', 'uniform',
]
print('CMD:', ' '.join(cmd))
if RUN_STEP3_ENSEMBLE:
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'Step-3 ensemble run failed: code={res.returncode}')
else:
    print('RUN_STEP3_ENSEMBLE=False (skipping execution).')


### 15.4 Feature-Side Upgrade (Engineered Video-Artifact Features)

This step adds engineered columns for video-only artifact sensitivity, then retrains:
- `video_motion_noise_ratio = mouth_flow_std / (abs(mouth_flow_mean)+eps)`
- `video_shape_noise_ratio = mouth_aspect_std / (abs(mouth_aspect_mean)+eps)`
- `video_temporal_instability = mouth_area_delta_std + mouth_asym_std`
- `video_detection_dropout = 1/(det_count+1)`
- `video_compression_proxy = (mouth_flow_std + mouth_area_delta_std + mouth_asym_std)/(det_count+1)`


In [ ]:
import numpy as np
import pandas as pd

RUN_BUILD_VIDEO_FEAT_CSV = False
SRC_CSV = BACKEND_ROOT / 'data/processed/causal_multimodal_dataset.csv'
DST_CSV = BACKEND_ROOT / 'data/processed/causal_multimodal_dataset_videofeatup_v1.csv'

if RUN_BUILD_VIDEO_FEAT_CSV:
    df = pd.read_csv(SRC_CSV)

    def num(col):
        if col not in df.columns:
            return pd.Series(np.zeros(len(df), dtype=float), index=df.index)
        return pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    eps = 1e-6
    mouth_flow_mean = num('mouth_flow_mean')
    mouth_flow_std = num('mouth_flow_std')
    mouth_aspect_mean = num('mouth_aspect_mean')
    mouth_aspect_std = num('mouth_aspect_std')
    mouth_area_delta_std = num('mouth_area_delta_std')
    mouth_asym_std = num('mouth_asym_std')
    det_count = num('det_count').clip(lower=0.0)

    df['video_motion_noise_ratio'] = mouth_flow_std / (mouth_flow_mean.abs() + eps)
    df['video_shape_noise_ratio'] = mouth_aspect_std / (mouth_aspect_mean.abs() + eps)
    df['video_temporal_instability'] = mouth_area_delta_std + mouth_asym_std
    df['video_detection_dropout'] = 1.0 / (det_count + 1.0)
    df['video_compression_proxy'] = (mouth_flow_std + mouth_area_delta_std + mouth_asym_std) / (det_count + 1.0)

    for c in [
        'video_motion_noise_ratio',
        'video_shape_noise_ratio',
        'video_temporal_instability',
        'video_detection_dropout',
        'video_compression_proxy',
    ]:
        s = pd.to_numeric(df[c], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0.0)
        df[c] = s.clip(lower=0.0).astype(float)

    df.to_csv(DST_CSV, index=False)
    print('Wrote:', DST_CSV)
    print(df[[
        'video_motion_noise_ratio','video_shape_noise_ratio','video_temporal_instability',
        'video_detection_dropout','video_compression_proxy'
    ]].describe().loc[['mean','std','min','max']])
else:
    print('RUN_BUILD_VIDEO_FEAT_CSV=False (skipping build).')
    print('Expected output path:', DST_CSV)


In [ ]:
import subprocess

RUN_STEP4_FEATUP_TRAIN = False
cmd = [
    str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
    '--processed-csv', 'data/processed/causal_multimodal_dataset_videofeatup_v1.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models', '--logs-dir', 'models/experiment_logs',
    '--run-prefix', 'fakeav_mrdf5cv_step4_videofeatup_v1',
    '--seed', '1337', '--split-seed', '42',
    '--n-splits', '5', '--val-size', '0.15', '--feature-profile', 'extended',
    '--epochs', '35', '--patience', '8', '--batch-size', '128',
    '--lr', '3e-4', '--weight-decay', '1e-4', '--causal-weight', '0.0',
    '--loss', 'focal', '--focal-alpha', '0.40', '--focal-gamma', '2.0',
    '--train-weight-application', 'both',
    '--enable-multitask', '--multitask-weight', '0.25',
    '--ranking-loss-weight', '0.20', '--ranking-margin', '0.20', '--ranking-max-pairs', '1024',
    '--pretrain-enable', '--pretrain-epochs', '12', '--pretrain-patience', '4',
    '--phase2-enable', '--phase2-rounds', '2', '--phase2-hardneg-source', 'train',
    '--phase2-hardpos-source', 'train', '--hard-negative-weight', '4.0', '--hard-positive-weight', '2.0',
    '--hardpos-scenario', 'audio_only_fake', '--phase2-scenario-focus', 'none', '--phase2-scenario-focus-weight', '1.0',
    '--eval-threshold-source', 'val_target', '--eval-threshold-priority', 'balanced_acc',
]
print('CMD:', ' '.join(cmd))
if RUN_STEP4_FEATUP_TRAIN:
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'Step-4 feature-upgrade run failed: code={res.returncode}')
else:
    print('RUN_STEP4_FEATUP_TRAIN=False (skipping execution).')


### 15.5 Result Aggregation Against Updated Targets

This cell consolidates run summaries and computes total shortfall against:
`Accuracy=0.98, Precision=0.87, Recall=0.88, F1=0.88, AUC=0.91`.


In [ ]:
import json
from pathlib import Path
from datetime import datetime
import pandas as pd

targets = UPDATED_TARGETS

logs_root = BACKEND_ROOT / 'models' / 'experiment_logs'

def latest_file(pattern):
    matches = sorted(logs_root.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[0] if matches else None

summary_paths = {
    # Accuracy-first runs
    'accfirst_a': latest_file('fakeav_mrdf5cv_accfirst_a_*/cv_summary.json'),
    'accfirst_b': latest_file('fakeav_mrdf5cv_accfirst_b_*/cv_summary.json'),
    'accfirst_c': latest_file('fakeav_mrdf5cv_accfirst_c_*/cv_summary.json'),
    'accfirst_d': latest_file('fakeav_mrdf5cv_accfirst_d_*/cv_summary.json'),

    # Earlier stepwise runs
    'step1_videoonly_focus': latest_file('fakeav_mrdf5cv_step1_videoonly_focus_*/cv_summary.json'),
    'step1_w25': latest_file('fakeav_mrdf5cv_step1_aucrecov_w25_*/cv_summary.json'),
    'step1_w30': latest_file('fakeav_mrdf5cv_step1_aucrecov_w30_*/cv_summary.json'),
    'step2_r020_a035': latest_file('fakeav_mrdf5cv_step2_r020_a035_*/cv_summary.json'),
    'step2_r020_a040': latest_file('fakeav_mrdf5cv_step2_r020_a040_*/cv_summary.json'),
    'step2_r030_a035': latest_file('fakeav_mrdf5cv_step2_r030_a035_*/cv_summary.json'),
    'step2_r030_a040': latest_file('fakeav_mrdf5cv_step2_r030_a040_*/cv_summary.json'),
    'step3_ensemble_top2_valauc': latest_file('fakeav_mrdf5cv_step3_top2valauc_multiseed_*/ensemble_summary.json'),
    'step4_videofeatup_v1': latest_file('fakeav_mrdf5cv_step4_videofeatup_v1_*/cv_summary.json'),
    'step5_ensemble_videofeatup': latest_file('fakeav_mrdf5cv_step5_videofeatup_multiseed_multiseed_*/ensemble_summary.json'),
    'step5_seed1337_videofeatup': latest_file('fakeav_mrdf5cv_step5_videofeatup_multiseed_s1337_*/cv_summary.json'),

    # Custom ensembles
    'custom_diverse_base3': latest_file('custom_diverse_ensemble_base3_*/ensemble_summary.json'),
    'custom_diverse_base3_plus_featup': latest_file('custom_diverse_ensemble_base3_plus_featup_*/ensemble_summary.json'),
}

rows = []
for name, p in summary_paths.items():
    if p is None or not Path(p).exists():
        continue
    obj = json.loads(Path(p).read_text())
    cv = obj.get('cv_metrics', {})
    row = {'run': name, 'summary_path': str(p)}
    for metric in ['test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc']:
        row[metric] = float(cv.get(metric, {}).get('mean', float('nan')))
    row['target_gap_total'] = sum(max(0.0, targets[m] - row[m]) for m in targets)
    rows.append(row)

if not rows:
    print('No summaries found.')
else:
    comp = pd.DataFrame(rows).sort_values(['target_gap_total', 'test_auc', 'test_acc'], ascending=[True, False, False]).reset_index(drop=True)
    display(comp[['run', 'target_gap_total', 'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc']])
    ts = datetime.now().strftime('%Y%m%d_%H%M')
    out = BACKEND_ROOT / 'models/experiment_logs' / f'comparison_nextsteps_{ts}_notebook_aggregated.csv'
    comp.to_csv(out, index=False)
    print('Saved:', out)


## 16) Accuracy-First Then Metrics-First Follow-Up

This section records the latest sequence:
1) accuracy-first retraining attempts (`accfirst_a/b/c/d`)
2) metrics-focused ensemble-policy sweep on top of step5 feature-upgraded seed runs.


In [ ]:
import subprocess

RUN_ACCFIRST_SWEEP = False
ACCFIRST_CONFIGS = [
    # run_prefix, loss, enable_multitask, multitask_w, ranking_w, phase2_rounds, hardneg_w
    ('fakeav_mrdf5cv_accfirst_a', 'focal', True, 0.25, 0.20, 2, 4.0),
    ('fakeav_mrdf5cv_accfirst_b', 'focal', False, 0.00, 0.00, 2, 4.0),
    ('fakeav_mrdf5cv_accfirst_c', 'bce',   False, 0.00, 0.00, 2, 4.0),
    ('fakeav_mrdf5cv_accfirst_d', 'focal', True, 0.25, 0.20, 1, 6.0),
]

for run_prefix, loss_name, en_mt, mt_w, rank_w, p2_rounds, hn_w in ACCFIRST_CONFIGS:
    cmd = [
        str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
        '--processed-csv', 'data/processed/causal_multimodal_dataset_videofeatup_v1.csv',
        '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
        '--models-dir', 'models', '--logs-dir', 'models/experiment_logs',
        '--run-prefix', run_prefix, '--seed', '1337', '--split-seed', '42',
        '--n-splits', '5', '--val-size', '0.15', '--feature-profile', 'extended',
        '--epochs', '35', '--patience', '8', '--batch-size', '128',
        '--lr', '3e-4', '--weight-decay', '1e-4', '--causal-weight', '0.0',
        '--loss', loss_name, '--train-weight-application', 'both',
        '--multitask-weight', str(float(mt_w)), '--ranking-loss-weight', str(float(rank_w)),
        '--ranking-margin', '0.20', '--ranking-max-pairs', '1024',
        '--pretrain-enable', '--pretrain-epochs', '12', '--pretrain-patience', '4',
        '--pretrain-scenario-focus', 'none', '--pretrain-scenario-focus-weight', '1.0',
        '--phase2-enable', '--phase2-hardneg-source', 'train', '--phase2-rounds', str(int(p2_rounds)),
        '--phase2-hardpos-source', 'train', '--hard-negative-weight', str(float(hn_w)), '--hard-positive-weight', '2.0',
        '--hardpos-scenario', 'audio_only_fake', '--phase2-scenario-focus', 'none', '--phase2-scenario-focus-weight', '1.0',
        '--eval-threshold-source', 'val_target', '--eval-threshold-priority', 'accuracy',
    ]
    if loss_name == 'focal':
        cmd.extend(['--focal-alpha', '0.40', '--focal-gamma', '2.0'])
    if en_mt:
        cmd.append('--enable-multitask')

    print('\nCMD:', ' '.join(cmd))
    if RUN_ACCFIRST_SWEEP:
        res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
        print(res.stdout[-4000:])
        if res.returncode != 0:
            print(res.stderr[-4000:])
            raise RuntimeError(f'Accuracy-first run failed for {run_prefix}: code={res.returncode}')

if not RUN_ACCFIRST_SWEEP:
    print('RUN_ACCFIRST_SWEEP=False (commands printed only).')


In [ ]:
# Metrics-focused follow-up: sweep ensemble policy on existing step5 seeds
import importlib.util
import numpy as np
import pandas as pd

RUN_STEP5_ENSEMBLE_POLICY_SWEEP = True

if RUN_STEP5_ENSEMBLE_POLICY_SWEEP:
    spec = importlib.util.spec_from_file_location('mrdf', BACKEND_ROOT / 'scripts' / 'run_fakeav_mrdf_5fold_cv.py')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    run_dirs = [
        BACKEND_ROOT / 'models/experiment_logs/fakeav_mrdf5cv_step5_videofeatup_multiseed_s42_20260312_233030',
        BACKEND_ROOT / 'models/experiment_logs/fakeav_mrdf5cv_step5_videofeatup_multiseed_s1337_20260312_233122',
        BACKEND_ROOT / 'models/experiment_logs/fakeav_mrdf5cv_step5_videofeatup_multiseed_s2026_20260312_233216',
    ]

    seed_frames = {}
    for d in run_dirs:
        fold_csv = d / 'fold_metrics.csv'
        if not fold_csv.exists():
            continue
        df = pd.read_csv(fold_csv)
        if 'error' in df.columns:
            df = df[df['error'].isna()].copy()
        if df.empty:
            continue
        seed_frames[int(df['seed'].iloc[0])] = df

    rows = []
    for rank_metric in ['val_auc', 'val_bal_acc', 'val_f1']:
        for top_k in [1, 2, 3]:
            for weighting in ['uniform', 'rank_metric']:
                fold_rows = []
                common_folds = set.intersection(*[set(df['fold']) for df in seed_frames.values()])
                for fold_name in sorted(common_folds):
                    cands = []
                    for _, df in seed_frames.items():
                        cands.append(df[df['fold'] == fold_name].iloc[0])
                    cands = sorted(cands, key=lambda r: float(r.get(rank_metric, float('-inf'))), reverse=True)[:top_k]

                    ref = None
                    val_probs_list, test_probs_list, weights = [], [], []
                    for r in cands:
                        val_raw, val_y, val_dom, val_probs, _ = mod._predict_probs(Path(r['model_dir']), Path(r['val_csv']), 'extended')
                        test_raw, test_y, test_dom, test_probs, _ = mod._predict_probs(Path(r['model_dir']), Path(r['test_csv']), 'extended')
                        vkey = mod._error_key_series(val_raw).astype(str).to_numpy()
                        tkey = mod._error_key_series(test_raw).astype(str).to_numpy()
                        if ref is None:
                            ref = (val_raw, val_y, val_dom, test_raw, test_y, test_dom, vkey, tkey)
                        else:
                            rv_raw, rv_y, rv_dom, rt_raw, rt_y, rt_dom, rv_key, rt_key = ref
                            if not (np.array_equal(val_y, rv_y) and np.array_equal(test_y, rt_y) and np.array_equal(vkey, rv_key) and np.array_equal(tkey, rt_key)):
                                continue

                        w = 1.0
                        if weighting == 'rank_metric':
                            w = float(r.get(rank_metric, 0.0))
                            if (not np.isfinite(w)) or (w <= 0.0):
                                w = 1e-6
                        val_probs_list.append(val_probs)
                        test_probs_list.append(test_probs)
                        weights.append(w)

                    if not val_probs_list:
                        continue

                    rv_raw, rv_y, rv_dom, rt_raw, rt_y, rt_dom, _, _ = ref
                    ens_val = np.average(np.vstack(val_probs_list), axis=0, weights=np.array(weights, dtype=float))
                    ens_test = np.average(np.vstack(test_probs_list), axis=0, weights=np.array(weights, dtype=float))
                    cal = mod.calibrate_threshold_to_targets(rv_y, ens_val, targets=mod.TARGETS, priority='balanced_acc')
                    thr = float(cal['threshold'])
                    tm = mod._metrics_from_probs(rt_raw, rt_y, rt_dom, ens_test, thr)
                    fold_rows.append(tm)

                if not fold_rows:
                    continue

                fr = pd.DataFrame(fold_rows)
                rows.append({
                    'rank_metric': rank_metric,
                    'top_k': int(top_k),
                    'weighting': weighting,
                    'test_acc': float(fr['acc'].mean()),
                    'test_prec': float(fr['prec'].mean()),
                    'test_rec': float(fr['rec'].mean()),
                    'test_f1': float(fr['f1'].mean()),
                    'test_auc': float(fr['auc'].mean()),
                })

    if not rows:
        print('No ensemble policy rows produced.')
    else:
        pol = pd.DataFrame(rows)
        pol['target_gap_total'] = (
            (0.98 - pol['test_acc']).clip(lower=0.0) +
            (0.87 - pol['test_prec']).clip(lower=0.0) +
            (0.88 - pol['test_rec']).clip(lower=0.0) +
            (0.88 - pol['test_f1']).clip(lower=0.0) +
            (0.91 - pol['test_auc']).clip(lower=0.0)
        )
        pol = pol.sort_values(['target_gap_total', 'test_auc', 'test_acc'], ascending=[True, False, False]).reset_index(drop=True)
        display(pol.head(12))
        out = BACKEND_ROOT / 'models/experiment_logs' / 'comparison_nextsteps_20260313_0033_step5_ensemble_policy_sweep.csv'
        pol.to_csv(out, index=False)
        print('Saved:', out)
else:
    print('RUN_STEP5_ENSEMBLE_POLICY_SWEEP=False (set True to execute).')


Current observation from this cycle: `step4_videofeatup_v1` remains best on total target gap among tested runs, while `step5` ensemble variants raise AUC but reduce accuracy.


## 17) Accuracy-First Weighting Ablation + Val-Accuracy Phase Policy


In [ ]:
from pathlib import Path
import subprocess
import sys

_cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
BACKEND_ROOT = next((p for p in _cands if (p / 'src').exists() and (p / 'scripts').exists()), _cands[-1])
PYTHON = BACKEND_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

RUN_WEIGHT_MODE_ABLATIONS = False  # set True to execute three full 5-fold runs

COMMON = [
    '--processed-csv', 'data/processed/causal_multimodal_dataset_videofeatup_v1.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models',
    '--logs-dir', 'models/experiment_logs',
    '--seed', '1337',
    '--split-seed', '42',
    '--n-splits', '5',
    '--val-size', '0.15',
    '--feature-profile', 'extended',
    '--epochs', '35',
    '--patience', '8',
    '--batch-size', '128',
    '--lr', '0.0003',
    '--weight-decay', '0.0001',
    '--causal-weight', '0.0',
    '--loss', 'focal',
    '--focal-alpha', '0.4',
    '--focal-gamma', '2.0',
    '--enable-multitask',
    '--multitask-weight', '0.25',
    '--ranking-loss-weight', '0.2',
    '--ranking-margin', '0.2',
    '--ranking-max-pairs', '1024',
    '--eval-threshold-source', 'val_target',
    '--eval-threshold-priority', 'accuracy',
    '--pretrain-enable',
    '--pretrain-epochs', '12',
    '--pretrain-patience', '4',
    '--pretrain-scenario-focus', 'none',
    '--pretrain-scenario-focus-weight', '1.0',
    '--phase2-enable',
    '--phase2-rounds', '2',
    '--phase2-hardneg-source', 'train',
    '--hard-negative-weight', '4.0',
    '--hardpos-scenario', 'audio_only_fake',
    '--phase2-scenario-focus', 'none',
    '--phase2-scenario-focus-weight', '1.0',
]

RUN_SPECS = [
    ('fakeav_mrdf5cv_accfirst_e_sampler', 'sampler'),
    ('fakeav_mrdf5cv_accfirst_f_loss', 'loss'),
    ('fakeav_mrdf5cv_accfirst_g_none', 'none'),
]

print('python:', PYTHON)
print('run_weight_mode_ablations:', RUN_WEIGHT_MODE_ABLATIONS)
for pfx, mode in RUN_SPECS:
    print('planned:', pfx, 'train-weight-application=', mode)


In [ ]:
if RUN_WEIGHT_MODE_ABLATIONS:
    for run_prefix, weight_mode in RUN_SPECS:
        cmd = [
            str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
            '--run-prefix', run_prefix,
            '--train-weight-application', weight_mode,
            *COMMON,
        ]
        print('\nRUNNING:', ' '.join(cmd))
        cp = subprocess.run(cmd, cwd=BACKEND_ROOT, check=False)
        print('return_code=', cp.returncode)
        if cp.returncode != 0:
            raise RuntimeError(f'Run failed: {run_prefix} ({weight_mode})')
else:
    print('Skipping training. Set RUN_WEIGHT_MODE_ABLATIONS=True to execute these runs.')


In [ ]:
import csv
import json
import pickle
import joblib
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import roc_auc_score

from src.modules.causal_fusion import CausalFusionNetworkV2
from src.training.train_cfn import (
    build_feature_matrix,
    resolve_feature_columns,
    infer_domain_labels,
    compute_domain_metrics,
)

if 'BACKEND_ROOT' not in globals() or not (Path(BACKEND_ROOT) / 'src').exists():
    _cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in _cands if (p / 'src').exists() and (p / 'scripts').exists()), _cands[-1])

EXP_ROOT = BACKEND_ROOT / 'models/experiment_logs'
EXP_ROOT.mkdir(parents=True, exist_ok=True)
TARGET_RUNS = [
    'fakeav_mrdf5cv_step5_videofeatup_multiseed_s1337_20260312_233122',
    'fakeav_mrdf5cv_accfirst_e_sampler_s1337_20260313_003754',
    'fakeav_mrdf5cv_accfirst_f_loss_s1337_20260313_003855',
    'fakeav_mrdf5cv_accfirst_g_none_s1337_20260313_004027',
]
TH_GRID = np.linspace(0.0, 1.0, 1001)
CACHE = {}

def _align_feature_dim(x: np.ndarray, target_dim: int) -> np.ndarray:
    if target_dim <= 0:
        return x
    cur = int(x.shape[1])
    if cur == target_dim:
        return x
    if cur > target_dim:
        return x[:, :target_dim]
    pad = np.zeros((x.shape[0], target_dim - cur), dtype=x.dtype)
    return np.concatenate([x, pad], axis=1)

def _scaler_expected_dim(scaler_obj) -> int | None:
    val = getattr(scaler_obj, 'n_features_in_', None)
    if val is None:
        return None
    try:
        out = int(val)
    except Exception:
        return None
    return out if out > 0 else None

def _load_probs(model_dir: Path, csv_path: Path):
    key = (str(model_dir), str(csv_path))
    if key in CACHE:
        return CACHE[key]

    df = pd.read_csv(csv_path)
    av_cols, phys_cols = resolve_feature_columns(
        df.columns,
        use_embeddings=True,
        profile='extended',
    )
    x_av = build_feature_matrix(df, av_cols, name='AV')
    x_phys = build_feature_matrix(df, phys_cols, name='PHYS')
    y = df['label'].astype(int).to_numpy()
    domains = infer_domain_labels(df)

    scaler_path = model_dir / 'cfn_scaler.pkl'
    if scaler_path.exists():
        scaler = None
        try:
            with open(scaler_path, 'rb') as f:
                scaler = pickle.load(f)
        except Exception:
            scaler = joblib.load(scaler_path)
        if isinstance(scaler, dict):
            if 'av' in scaler:
                exp = _scaler_expected_dim(scaler['av'])
                if exp is not None:
                    x_av = _align_feature_dim(x_av, exp)
                x_av = scaler['av'].transform(x_av)
            if 'phys' in scaler:
                exp = _scaler_expected_dim(scaler['phys'])
                if exp is not None:
                    x_phys = _align_feature_dim(x_phys, exp)
                x_phys = scaler['phys'].transform(x_phys)

    state = torch.load(model_dir / 'cfn_emb.pth', map_location='cpu')
    av_dim = int(state.get('av_branch.0.weight', torch.empty(0)).shape[1])
    phys_dim = int(state.get('physical_branch.0.weight', torch.empty(0)).shape[1])
    if av_dim <= 0:
        av_dim = int(x_av.shape[1])
    if phys_dim <= 0:
        phys_dim = int(x_phys.shape[1])
    x_av = _align_feature_dim(x_av, av_dim)
    x_phys = _align_feature_dim(x_phys, phys_dim)

    model = CausalFusionNetworkV2(av_dim=av_dim, phys_dim=phys_dim)
    model.load_state_dict(state, strict=False)
    model.eval()
    with torch.no_grad():
        probs = model(
            torch.from_numpy(x_av.astype(np.float32)),
            torch.from_numpy(x_phys.astype(np.float32)),
        ).squeeze(1).cpu().numpy()

    CACHE[key] = (y, domains, probs)
    return CACHE[key]

def _best_threshold_by_val_acc(y, domains, probs) -> tuple[float, tuple[float, float, float]]:
    best_key = (-1.0, -1.0, -1.0)
    best_t = 0.5
    for t in TH_GRID:
        m = compute_domain_metrics(y, probs, domains, threshold=float(t))['overall']
        key = (float(m['acc']), float(m['bal_acc']), float(m['f1']))
        if key > best_key:
            best_key = key
            best_t = float(t)
    return best_t, best_key

def _test_metrics(y, domains, probs, threshold: float) -> dict[str, float]:
    m = compute_domain_metrics(y, probs, domains, threshold=threshold)['overall']
    return {
        'acc': float(m['acc']),
        'prec': float(m['prec']),
        'rec': float(m['rec']),
        'f1': float(m['f1']),
        'auc': float(roc_auc_score(y, probs)),
        'bal_acc': float(m['bal_acc']),
    }

rows = []
for run in TARGET_RUNS:
    cv_path = EXP_ROOT / run / 'cv_summary.json'
    if cv_path.exists():
        cv = json.loads(cv_path.read_text())
        m = cv.get('cv_metrics', {})
        rows.append({
            'run': run,
            'policy': 'official_cv_summary',
            'acc': float(m.get('test_acc', {}).get('mean', np.nan)),
            'prec': float(m.get('test_prec', {}).get('mean', np.nan)),
            'rec': float(m.get('test_rec', {}).get('mean', np.nan)),
            'f1': float(m.get('test_f1', {}).get('mean', np.nan)),
            'auc': float(m.get('test_auc', {}).get('mean', np.nan)),
            'bal_acc': float(m.get('test_bal_acc', {}).get('mean', np.nan)),
        })

    fold_csv = EXP_ROOT / run / 'fold_metrics.csv'
    if not fold_csv.exists():
        continue
    with open(fold_csv, 'r') as f:
        folds = list(csv.DictReader(f))
    fold_metrics = []
    for fold in folds:
        best_rank = None
        chosen = None
        for key in ['p1_model_dir', 'p2_r01_model_dir', 'p2_r02_model_dir']:
            model_dir = Path(fold.get(key, ''))
            if not model_dir.exists() or not (model_dir / 'cfn_emb.pth').exists():
                continue
            y_val, d_val, p_val = _load_probs(model_dir, Path(fold['val_csv']))
            thr, val_key = _best_threshold_by_val_acc(y_val, d_val, p_val)
            val_auc = float(roc_auc_score(y_val, p_val))
            rank = (val_key[0], val_key[1], val_key[2], val_auc)
            if best_rank is None or rank > best_rank:
                best_rank = rank
                chosen = (model_dir, thr)

        if chosen is None:
            continue

        y_test, d_test, p_test = _load_probs(chosen[0], Path(fold['test_csv']))
        fold_metrics.append(_test_metrics(y_test, d_test, p_test, chosen[1]))

    if fold_metrics:
        rows.append({
            'run': run,
            'policy': 'best_phase_valacc_threshold',
            'acc': float(np.mean([m['acc'] for m in fold_metrics])),
            'prec': float(np.mean([m['prec'] for m in fold_metrics])),
            'rec': float(np.mean([m['rec'] for m in fold_metrics])),
            'f1': float(np.mean([m['f1'] for m in fold_metrics])),
            'auc': float(np.mean([m['auc'] for m in fold_metrics])),
            'bal_acc': float(np.mean([m['bal_acc'] for m in fold_metrics])),
        })

res = pd.DataFrame(rows)
required_cols = ['acc', 'auc', 'f1']
if res.empty:
    print('No rows were produced. Check TARGET_RUNS paths and generated artifacts.')
    display(res)
else:
    missing = [c for c in required_cols if c not in res.columns]
    if missing:
        print(f'Missing expected metric columns: {missing}')
        display(res)
    else:
        res = res.sort_values(required_cols, ascending=False).reset_index(drop=True)
        display(res)

out_csv = EXP_ROOT / 'comparison_nextsteps_20260313_weightmode_phase_policy.csv'
res.to_csv(out_csv, index=False)
print('saved:', out_csv)


Accuracy-first result from this section: `best_phase_valacc_threshold` on `step5_videofeatup_multiseed_s1337` reached ~`0.7881` mean CV accuracy (up from `0.7856` official), while sampler-only weighting increased AUC but did not increase official accuracy.


## 18) Official Accuracy Policy + Focused Step-6 Sweep


In [ ]:
from pathlib import Path
import subprocess
import itertools
import sys

_cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
BACKEND_ROOT = next((p for p in _cands if (p / 'src').exists() and (p / 'scripts').exists()), _cands[-1])
PYTHON = BACKEND_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

# Official accuracy-first policy in CV runner:
#   --eval-threshold-source val_acc
#   --phase-selection-priority accuracy
RUN_STEP6_ACCPOLICY_SWEEP = False  # set True to execute full 12-run grid

WEIGHT_MODES = ['both', 'none']
PHASE2_ROUNDS = [2, 3]
HARD_NEG_WEIGHTS = [4.0, 6.0, 8.0]

COMMON = [
    '--processed-csv', 'data/processed/causal_multimodal_dataset_videofeatup_v1.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models',
    '--logs-dir', 'models/experiment_logs',
    '--seed', '1337',
    '--split-seed', '42',
    '--n-splits', '5',
    '--val-size', '0.15',
    '--feature-profile', 'extended',
    '--epochs', '35',
    '--patience', '8',
    '--batch-size', '128',
    '--lr', '0.0003',
    '--weight-decay', '0.0001',
    '--causal-weight', '0.0',
    '--loss', 'focal',
    '--focal-alpha', '0.4',
    '--focal-gamma', '2.0',
    '--enable-multitask',
    '--multitask-weight', '0.25',
    '--ranking-loss-weight', '0.2',
    '--ranking-margin', '0.2',
    '--ranking-max-pairs', '1024',
    '--eval-threshold-source', 'val_acc',
    '--eval-threshold-priority', 'accuracy',
    '--phase-selection-priority', 'accuracy',
    '--pretrain-enable',
    '--pretrain-epochs', '12',
    '--pretrain-patience', '4',
    '--pretrain-scenario-focus', 'none',
    '--pretrain-scenario-focus-weight', '1.0',
    '--phase2-enable',
    '--phase2-hardneg-source', 'train',
    '--hardpos-scenario', 'audio_only_fake',
    '--phase2-scenario-focus', 'none',
    '--phase2-scenario-focus-weight', '1.0',
]

print('python:', PYTHON)
print('run_step6_accpolicy_sweep:', RUN_STEP6_ACCPOLICY_SWEEP)
print('grid_size:', len(WEIGHT_MODES) * len(PHASE2_ROUNDS) * len(HARD_NEG_WEIGHTS))


In [ ]:
import json
import pandas as pd

rows = []
if RUN_STEP6_ACCPOLICY_SWEEP:
    for wm, r2, hn in itertools.product(WEIGHT_MODES, PHASE2_ROUNDS, HARD_NEG_WEIGHTS):
        run_prefix = f'fakeav_mrdf5cv_step6_accpolicy_wm{wm}_r{int(r2)}_hn{int(hn)}'
        cmd = [
            str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
            '--run-prefix', run_prefix,
            '--train-weight-application', wm,
            '--phase2-rounds', str(int(r2)),
            '--hard-negative-weight', str(float(hn)),
            *COMMON,
        ]
        print('\nRUNNING:', run_prefix)
        cp = subprocess.run(cmd, cwd=BACKEND_ROOT, check=False)
        if cp.returncode != 0:
            rows.append({
                'run_prefix': run_prefix,
                'weight_mode': wm,
                'phase2_rounds': int(r2),
                'hard_negative_weight': float(hn),
                'return_code': int(cp.returncode),
            })
            continue

        # Resolve latest run tag for this prefix
        cands = sorted((BACKEND_ROOT / 'models/experiment_logs').glob(f'{run_prefix}_s1337_*'))
        run_dir = cands[-1] if cands else None
        cv_path = (run_dir / 'cv_summary.json') if run_dir else None
        metric_row = {
            'run_prefix': run_prefix,
            'run_tag': (run_dir.name if run_dir else None),
            'weight_mode': wm,
            'phase2_rounds': int(r2),
            'hard_negative_weight': float(hn),
            'return_code': int(cp.returncode),
        }
        if cv_path and cv_path.exists():
            cv = json.loads(cv_path.read_text())
            cm = cv.get('cv_metrics', {})
            metric_row.update({
                'acc': float((cm.get('test_acc') or {}).get('mean', float('nan'))),
                'prec': float((cm.get('test_prec') or {}).get('mean', float('nan'))),
                'rec': float((cm.get('test_rec') or {}).get('mean', float('nan'))),
                'f1': float((cm.get('test_f1') or {}).get('mean', float('nan'))),
                'auc': float((cm.get('test_auc') or {}).get('mean', float('nan'))),
                'bal_acc': float((cm.get('test_bal_acc') or {}).get('mean', float('nan'))),
            })
        rows.append(metric_row)

    sweep_df = pd.DataFrame(rows)
    out_csv = BACKEND_ROOT / 'models/experiment_logs/comparison_nextsteps_20260313_step6_accpolicy_sweep.csv'
    sweep_df.to_csv(out_csv, index=False)
    print('saved:', out_csv)
else:
    print('Skipping run execution. Set RUN_STEP6_ACCPOLICY_SWEEP=True to run the full grid.')


In [ ]:
import pandas as pd

sweep_csv = BACKEND_ROOT / 'models/experiment_logs/comparison_nextsteps_20260313_step6_accpolicy_sweep.csv'
if sweep_csv.exists():
    sweep_df = pd.read_csv(sweep_csv)
    if 'return_code' in sweep_df.columns:
        rank_df = sweep_df[sweep_df['return_code'] == 0].copy()
    else:
        rank_df = sweep_df.copy()

    metric_cols = [c for c in ['acc', 'auc', 'f1'] if c in rank_df.columns]
    if metric_cols:
        rank_df = rank_df.sort_values(metric_cols, ascending=[False] * len(metric_cols)).reset_index(drop=True)

    show_cols = [c for c in ['run_tag', 'weight_mode', 'phase2_rounds', 'hard_negative_weight', 'acc', 'prec', 'rec', 'f1', 'auc', 'bal_acc'] if c in rank_df.columns]
    if show_cols:
        display(rank_df[show_cols].head(20))
    else:
        display(rank_df.head(20))

    if not rank_df.empty:
        best = rank_df.iloc[0]
        print(
            f"BEST STEP-6 ACC POLICY: run={best.get('run_tag', 'N/A')} | "
            f"acc={float(best.get('acc', float('nan'))):.6f} "
            f"prec={float(best.get('prec', float('nan'))):.6f} "
            f"rec={float(best.get('rec', float('nan'))):.6f} "
            f"f1={float(best.get('f1', float('nan'))):.6f} "
            f"auc={float(best.get('auc', float('nan'))):.6f} "
            f"bal_acc={float(best.get('bal_acc', float('nan'))):.6f}"
        )
else:
    print('Missing sweep CSV:', sweep_csv)


Latest observed best from this Step-6 sweep: accuracy reached about `0.7921` with `weight_mode=none` and `phase2_rounds=3` (hn weight 4/6/8 tied), which is higher than the previous ~`0.7881` post-hoc accuracy policy.


## 19) Multi-Metric Follow-Up (Post-Hoc + Multi-Seed)


In [ ]:
import pandas as pd

POSTHOC_CSV = BACKEND_ROOT / 'models/experiment_logs/comparison_nextsteps_20260313_posthoc_threshold_policies.csv'
if POSTHOC_CSV.exists():
    posthoc_df = pd.read_csv(POSTHOC_CSV)
    sort_cols = [c for c in ['acc', 'auc', 'f1'] if c in posthoc_df.columns]
    if sort_cols:
        posthoc_df = posthoc_df.sort_values(sort_cols, ascending=[False] * len(sort_cols))
    display(posthoc_df.head(20))
else:
    print('Missing:', POSTHOC_CSV)


In [ ]:
# Optional: rerun 3-seed multi-seed + ensemble for the best balanced config
RUN_STEP7_MULTI_SEED = False

if RUN_STEP7_MULTI_SEED:
    cmd = [
        str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
        '--processed-csv', 'data/processed/causal_multimodal_dataset_videofeatup_v1.csv',
        '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
        '--models-dir', 'models',
        '--logs-dir', 'models/experiment_logs',
        '--run-prefix', 'fakeav_mrdf5cv_step7_accpolicy_wmboth_r2_hn8_multiseed',
        '--seed', '1337',
        '--seed-list', '42,1337,2026',
        '--split-seed', '42',
        '--n-splits', '5',
        '--val-size', '0.15',
        '--feature-profile', 'extended',
        '--epochs', '35',
        '--patience', '8',
        '--batch-size', '128',
        '--lr', '0.0003',
        '--weight-decay', '0.0001',
        '--causal-weight', '0.0',
        '--loss', 'focal',
        '--focal-alpha', '0.4',
        '--focal-gamma', '2.0',
        '--train-weight-application', 'both',
        '--enable-multitask',
        '--multitask-weight', '0.25',
        '--ranking-loss-weight', '0.2',
        '--ranking-margin', '0.2',
        '--ranking-max-pairs', '1024',
        '--eval-threshold-source', 'val_acc',
        '--eval-threshold-priority', 'accuracy',
        '--phase-selection-priority', 'accuracy',
        '--pretrain-enable',
        '--pretrain-epochs', '12',
        '--pretrain-patience', '4',
        '--pretrain-scenario-focus', 'none',
        '--pretrain-scenario-focus-weight', '1.0',
        '--phase2-enable',
        '--phase2-rounds', '2',
        '--phase2-hardneg-source', 'train',
        '--hard-negative-weight', '8.0',
        '--hardpos-scenario', 'audio_only_fake',
        '--phase2-scenario-focus', 'none',
        '--phase2-scenario-focus-weight', '1.0',
        '--ensemble-enable',
        '--ensemble-top-k', '2',
        '--ensemble-rank-metric', 'val_auc',
        '--ensemble-weighting', 'rank_metric',
    ]
    print('RUNNING:', ' '.join(cmd))
    cp = subprocess.run(cmd, cwd=BACKEND_ROOT, check=False)
    print('return_code=', cp.returncode)
else:
    print('Skipping multi-seed rerun. Set RUN_STEP7_MULTI_SEED=True to execute.')


In [ ]:
SUMMARY_CSV = BACKEND_ROOT / 'models/experiment_logs/comparison_nextsteps_20260313_step8_summary.csv'
if SUMMARY_CSV.exists():
    summary_df = pd.read_csv(SUMMARY_CSV)
    sort_cols = [c for c in ['acc', 'auc', 'f1'] if c in summary_df.columns]
    if sort_cols:
        summary_df = summary_df.sort_values(sort_cols, ascending=[False] * len(sort_cols))
    display(summary_df)
else:
    print('Missing:', SUMMARY_CSV)


Current best official single-seed checkpoint is `step8_accpolicy_densegrid_wmboth_r2_hn8` with about `acc=0.7936`, `prec=0.8480`, `rec=0.8844`, `f1=0.8654`, `auc=0.8196`.
For AUC-focused comparison, the top-2 multi-seed ensemble increased AUC to about `0.8236` but reduced accuracy.


## 20) Advanced Step-1..7 Protocol (New)

This section adds runnable cells for the requested improvements:

1. Step-1: weighted loss + weighted sampler + silence trim.
2. Step-2: Wav2Vec2 Base staged fine-tuning (frozen first, then unfreeze last 2-3 layers).
3. Step-3: EfficientNet-B4 pretrained visual embedding augmentation.
4. Step-4: Cross-modal attention fusion in CFN (already in model; enabled in training path).
5. Step-5: Composite loss with dedicated causal-breach supervision.
6. Step-6: Lip ROI extraction as a third visual stream.
7. Step-7: Staged training protocol (pretrain + phase2 hard-example rounds).

All cells below are idempotent and include guard flags for heavy compute.


In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess
from datetime import datetime

# Robust root/python bootstrap to avoid NameError and wrong cwd roots when running out of order.
if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'src').exists() or not (BACKEND_ROOT / 'scripts').exists():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        Path.cwd().resolve() / 'backend',
    ]
    BACKEND_ROOT = next((p for p in candidates if (p / 'src').exists() and (p / 'scripts').exists()), candidates[-1])

PYTHON = BACKEND_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

RUN_STEP1_TRIM = True
RUN_STEP2_W2V2_FINETUNE = True
RUN_STEP3_EFFNET_AUGMENT = True
RUN_STEP4_7_ADVANCED_MRDF = True

ADV_RUN_PREFIX = 'fakeav_mrdf5cv_step20_adv'
ADV_TS = datetime.now().strftime('%Y%m%d_%H%M%S')

INPUT_PROCESSED_CSV = BACKEND_ROOT / 'data/processed/causal_multimodal_dataset.csv'
ADV_PROCESSED_CSV = BACKEND_ROOT / 'data/processed/causal_multimodal_dataset_effnet_w2v2.csv'

print('BACKEND_ROOT =', BACKEND_ROOT)
print('PYTHON =', PYTHON)


In [ ]:
# Step-1: silence-shortcut mitigation (trim first 100ms).
from pathlib import Path
if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'scripts' / 'trim_fakeav_audio_head.py').exists():
    _cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in _cands if (p / 'scripts' / 'trim_fakeav_audio_head.py').exists()), _cands[-1])
TRIM_MANIFEST = BACKEND_ROOT / 'data/processed/fakeav_audio_trim_manifest.csv'
if RUN_STEP1_TRIM:
    import sys
    if 'PYTHON' in globals():
        PYTHON = Path(PYTHON)
    if 'PYTHON' not in globals() or not PYTHON.exists():
        py_cand = BACKEND_ROOT / '.venv' / 'bin' / 'python'
        PYTHON = py_cand if py_cand.exists() else Path(sys.executable)
    script_path = BACKEND_ROOT / 'scripts/trim_fakeav_audio_head.py'
    cmd = [
        str(PYTHON), str(script_path),
        '--input-root', 'data/raw/fakeavceleb',
        '--trim-seconds', '0.10',
        '--in-place',
        '--manifest-csv', str(TRIM_MANIFEST.relative_to(BACKEND_ROOT)),
    ]
    print('CMD:', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'trim_fakeav_audio_head.py failed: code={res.returncode}')
else:
    print('RUN_STEP1_TRIM=False (skip).')

if TRIM_MANIFEST.exists():
    import pandas as pd
    m = pd.read_csv(TRIM_MANIFEST)
    print('Trim status counts:')
    print(m['status'].value_counts(dropna=False).head(10))
    print('Manifest:', TRIM_MANIFEST)


In [ ]:
# Step-2: Wav2Vec2 Base staged fine-tune (freeze backbone, then unfreeze last 2-3 layers).
W2V2_CKPT = BACKEND_ROOT / 'models/wav2vec2_base_fakeav_ft.pt'
if RUN_STEP2_W2V2_FINETUNE:
    cmd = [
        str(PYTHON), 'scripts/finetune_wav2vec2_base_fakeav.py',
        '--processed-csv', str(INPUT_PROCESSED_CSV.relative_to(BACKEND_ROOT)),
        '--fakeav-root', 'data/raw/fakeavceleb',
        '--output-checkpoint', str(W2V2_CKPT.relative_to(BACKEND_ROOT)),
        '--epochs', '6',
        '--freeze-epochs', '2',
        '--unfreeze-last-layers', '3',
        '--batch-size', '4',
        '--lr-head', '1e-3',
        '--lr-backbone', '1e-5',
        '--max-seconds', '6.0',
        '--sample-rate', '16000',
    ]
    print('CMD:', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'finetune_wav2vec2_base_fakeav.py failed: code={res.returncode}')
else:
    print('RUN_STEP2_W2V2_FINETUNE=False (skip).')

if W2V2_CKPT.exists():
    print('W2V2 checkpoint ready:', W2V2_CKPT)
    print('To use in extraction/training session:')
    print('  export CFN_W2V2_FINETUNED_PATH="%s"' % W2V2_CKPT)


In [ ]:
# Step-3 + Step-6 preprocessing: EfficientNet-B4 features + Lip ROI stream columns.
if RUN_STEP3_EFFNET_AUGMENT:
    cmd = [
        str(PYTHON), 'scripts/augment_processed_with_effnet_w2v2.py',
        '--input-csv', str(INPUT_PROCESSED_CSV.relative_to(BACKEND_ROOT)),
        '--output-csv', str(ADV_PROCESSED_CSV.relative_to(BACKEND_ROOT)),
        '--fakeav-root', 'data/raw/fakeavceleb',
        '--frame-stride', '6',
        '--sample-frames', '12',
        '--audio-seconds', '6.0',
    ]
    print('CMD:', ' '.join(cmd))
    env = os.environ.copy()
    # If Step-2 produced checkpoint, propagate it so wav2vec2_base_ft_emb reflects fine-tuned weights.
    if (BACKEND_ROOT / 'models/wav2vec2_base_fakeav_ft.pt').exists():
        env['CFN_W2V2_FINETUNED_PATH'] = str(BACKEND_ROOT / 'models/wav2vec2_base_fakeav_ft.pt')
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True, env=env)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'augment_processed_with_effnet_w2v2.py failed: code={res.returncode}')
else:
    print('RUN_STEP3_EFFNET_AUGMENT=False (skip).')

if ADV_PROCESSED_CSV.exists():
    import pandas as pd
    _tmp = pd.read_csv(ADV_PROCESSED_CSV, nrows=5)
    print('Advanced processed CSV:', ADV_PROCESSED_CSV)
    print('Has columns:', [c for c in ['effnet_b4_face_emb', 'lip_roi_emb', 'wav2vec2_base_ft_emb'] if c in _tmp.columns])


In [ ]:
# Step-4 + Step-5 + Step-7:
#   - Cross-modal attention CFN (already in model)
#   - Composite loss (causal breach supervision)
#   - Staged protocol (pretrain + phase2)

processed_csv_for_run = ADV_PROCESSED_CSV if ADV_PROCESSED_CSV.exists() else INPUT_PROCESSED_CSV

if RUN_STEP4_7_ADVANCED_MRDF:
    cmd = [
        str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
        '--processed-csv', str(processed_csv_for_run.relative_to(BACKEND_ROOT)),
        '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
        '--models-dir', 'models',
        '--logs-dir', 'models/experiment_logs',
        '--run-prefix', ADV_RUN_PREFIX,
        '--seed', '1337',
        '--seed-list', '42,1337,2026',
        '--split-seed', '42',
        '--n-splits', '5',
        '--val-size', '0.15',
        '--feature-profile', 'extended',
        '--epochs', '35',
        '--patience', '8',
        '--batch-size', '128',
        '--lr', '3e-4',
        '--weight-decay', '1e-4',
        '--loss', 'focal',
        '--focal-alpha', '0.40',
        '--focal-gamma', '2.0',
        '--train-weight-application', 'both',
        '--enable-multitask',
        '--multitask-weight', '0.25',
        '--ranking-loss-weight', '0.20',
        '--ranking-margin', '0.20',
        '--ranking-max-pairs', '1024',
        '--enable-lip-stream',
        '--causal-breach-loss-weight', '0.20',
        '--causal-breach-target', 'heuristic',
        '--pretrain-enable',
        '--pretrain-epochs', '12',
        '--pretrain-patience', '4',
        '--phase2-enable',
        '--phase2-rounds', '2',
        '--phase2-hardneg-source', 'train',
        '--phase2-hardpos-source', 'train',
        '--phase2-use-hard-positives',
        '--hard-negative-weight', '4.0',
        '--hard-positive-weight', '2.0',
        '--hardpos-scenario', 'audio_only_fake',
        '--phase2-scenario-focus', 'none',
        '--phase2-scenario-focus-weight', '1.0',
        '--eval-threshold-source', 'val_target',
        '--eval-threshold-priority', 'balanced_acc',
        '--ensemble-enable',
        '--ensemble-top-k', '3',
        '--ensemble-rank-metric', 'val_auc',
        '--ensemble-weighting', 'uniform',
    ]
    print('CMD:', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'run_fakeav_mrdf_5fold_cv.py failed: code={res.returncode}')
else:
    print('RUN_STEP4_7_ADVANCED_MRDF=False (skip).')


In [ ]:
# Robust result scan for this advanced step section.
import pandas as pd
import json
from pathlib import Path

logs_root = BACKEND_ROOT / 'models/experiment_logs'
run_dirs = sorted([p for p in logs_root.glob(f'{ADV_RUN_PREFIX}*') if p.is_dir()], key=lambda p: p.stat().st_mtime)
if not run_dirs:
    print('No run folders found for prefix:', ADV_RUN_PREFIX)
else:
    latest = run_dirs[-1]
    print('Latest run dir:', latest)
    fold_csv = latest / 'fold_metrics.csv'
    summary_json = latest / 'cv_summary.json'

    if fold_csv.exists():
        fold = pd.read_csv(fold_csv)
        display_cols = [c for c in [
            'fold',
            'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc',
            'test_f1_real', 'test_f1_fake',
            'test_auc_manip_audio_only', 'test_auc_manip_video_only', 'test_auc_manip_both_fake',
            'error'
        ] if c in fold.columns]
        display(fold[display_cols])
    else:
        print('Missing fold_metrics.csv:', fold_csv)

    if summary_json.exists():
        obj = json.loads(summary_json.read_text())
        means = obj.get('cv_mean', {})
        print('CV Means:')
        print({k: means.get(k) for k in [
            'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc'
        ]})
    else:
        print('Missing cv_summary.json:', summary_json)


## 21) Specificity-Recovery Sweep (Step-21)

This section runs the 6-config sweep focused on reducing false positives while keeping recall high:

- `causal_breach_loss_weight` in `{0.05, 0.10}`
- `hard_negative_weight` in `{6, 8, 10}`
- `eval-threshold-source=val_acc`, `eval-threshold-priority=accuracy`, `phase-selection-priority=accuracy`

Gate for promotion to multiseed/ensemble:

- Must beat Step-9 baseline on **both** `acc` and `auc`
- Must keep `rec >= 0.88`


In [ ]:
from pathlib import Path
import subprocess
import json
from datetime import datetime
import pandas as pd

# Robust root/python resolution.
if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'scripts').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'scripts').exists() and (p / 'src').exists()), cands[-1])

if 'PYTHON' in globals():
    PYTHON = Path(PYTHON)
if 'PYTHON' not in globals() or not PYTHON.exists():
    py = BACKEND_ROOT / '.venv' / 'bin' / 'python'
    import sys
    PYTHON = py if py.exists() else Path(sys.executable)

RUN_STEP21_SPEC_SWEEP = False  # set True to execute full 6-run sweep
STEP21_PREFIX = 'fakeav_mrdf5cv_step21_specrecov'
STEP21_GRID = [(0.05, 6.0), (0.05, 8.0), (0.05, 10.0), (0.10, 6.0), (0.10, 8.0), (0.10, 10.0)]
STEP21_OUT_CSV = BACKEND_ROOT / 'models/experiment_logs' / f'comparison_nextsteps_{datetime.now().strftime("%Y%m%d_%H%M%S")}_step21_spec_recovery.csv'

COMMON = [
    '--processed-csv', 'data/processed/causal_multimodal_dataset_effnet_w2v2.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models',
    '--logs-dir', 'models/experiment_logs',
    '--seed', '1337',
    '--split-seed', '42',
    '--n-splits', '5',
    '--val-size', '0.15',
    '--feature-profile', 'extended',
    '--epochs', '35',
    '--patience', '8',
    '--batch-size', '128',
    '--lr', '0.0003',
    '--weight-decay', '0.0001',
    '--loss', 'focal',
    '--focal-alpha', '0.40',
    '--focal-gamma', '2.0',
    '--train-weight-application', 'both',
    '--enable-multitask',
    '--multitask-weight', '0.25',
    '--ranking-loss-weight', '0.20',
    '--ranking-margin', '0.20',
    '--ranking-max-pairs', '1024',
    '--enable-lip-stream',
    '--causal-breach-target', 'heuristic',
    '--pretrain-enable',
    '--pretrain-epochs', '12',
    '--pretrain-patience', '4',
    '--phase2-enable',
    '--phase2-rounds', '2',
    '--phase2-hardneg-source', 'train',
    '--phase2-hardpos-source', 'train',
    '--phase2-use-hard-positives',
    '--hard-positive-weight', '2.0',
    '--hardpos-scenario', 'audio_only_fake',
    '--phase2-scenario-focus', 'none',
    '--phase2-scenario-focus-weight', '1.0',
    '--eval-threshold-source', 'val_acc',
    '--eval-threshold-priority', 'accuracy',
    '--phase-selection-priority', 'accuracy',
]

if RUN_STEP21_SPEC_SWEEP:
    rows = []
    for cw, hn in STEP21_GRID:
        run_prefix = f'{STEP21_PREFIX}_cw{str(cw).replace(".", "p")}_hn{int(hn)}'
        cmd = [
            str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
            '--run-prefix', run_prefix,
            '--causal-breach-loss-weight', str(float(cw)),
            '--hard-negative-weight', str(float(hn)),
            *COMMON,
        ]
        print('\nRUN:', run_prefix)
        cp = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True)

        row = {
            'run_prefix': run_prefix,
            'causal_breach_loss_weight': float(cw),
            'hard_negative_weight': float(hn),
            'return_code': int(cp.returncode),
        }

        cands = sorted((BACKEND_ROOT / 'models/experiment_logs').glob(f'{run_prefix}_s1337_*'))
        run_dir = cands[-1] if cands else None
        row['run_tag'] = run_dir.name if run_dir else None

        if run_dir and (run_dir / 'cv_summary.json').exists():
            obj = json.loads((run_dir / 'cv_summary.json').read_text())
            cm = obj.get('cv_metrics', {})
            for k in ['test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real']:
                v = cm.get(k, {})
                row[k] = float(v.get('mean')) if isinstance(v, dict) and v.get('mean') is not None else None

        rows.append(row)
        pd.DataFrame(rows).sort_values([c for c in ['test_acc', 'test_auc', 'test_f1'] if c in pd.DataFrame(rows).columns], ascending=False).to_csv(STEP21_OUT_CSV, index=False)

    print('saved:', STEP21_OUT_CSV)
else:
    print('RUN_STEP21_SPEC_SWEEP=False (skip)')
    latest = sorted((BACKEND_ROOT / 'models/experiment_logs').glob('comparison_nextsteps_*_step21_spec_recovery.csv'))
    if latest:
        print('latest existing:', latest[-1])


In [ ]:
import pandas as pd
import json
from pathlib import Path

logs_root = BACKEND_ROOT / 'models/experiment_logs'
step21_csvs = sorted(logs_root.glob('comparison_nextsteps_*_step21_spec_recovery.csv'))
if not step21_csvs:
    print('No step21 sweep CSV found.')
else:
    s21 = pd.read_csv(step21_csvs[-1])
    for c in ['test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real']:
        if c in s21.columns:
            s21[c] = pd.to_numeric(s21[c], errors='coerce')
    rank_cols = [c for c in ['test_acc', 'test_auc', 'test_f1'] if c in s21.columns]
    if rank_cols:
        s21 = s21.sort_values(rank_cols, ascending=[False] * len(rank_cols)).reset_index(drop=True)
    display_cols = [c for c in ['run_prefix', 'run_tag', 'return_code', 'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real'] if c in s21.columns]
    display(s21[display_cols])

    baseline_path = logs_root / 'fakeav_mrdf5cv_step9_densegrid_wmboth_r2_hn8_multiseed_s1337_20260313_131255/cv_summary.json'
    if baseline_path.exists() and not s21.empty:
        b = json.loads(baseline_path.read_text())
        bm = b.get('cv_metrics', {})
        bvals = {k: float((bm.get(k) or {}).get('mean')) for k in ['test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc']}

        best = s21.iloc[0]
        print('\nBest Step-21 candidate:', best.get('run_tag', 'N/A'))
        for k in ['test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc']:
            if k in s21.columns:
                v = float(best.get(k, float('nan')))
                print(f'  {k}: {v:.6f} (delta vs baseline {v - bvals[k]:+.6f})')

        gate = (
            float(best.get('test_acc', -1)) > bvals['test_acc']
            and float(best.get('test_auc', -1)) > bvals['test_auc']
            and float(best.get('test_rec', -1)) >= 0.88
        )
        print('\nPromotion gate (acc↑ and auc↑ and rec>=0.88):', gate)
        if gate:
            print('Proceed to multiseed+ensemble on this Step-21 config.')
        else:
            print('Do not promote Step-21 sweep results; keep Step-9 baseline.')


## 22) Specificity-Focused Sweep (Step-22, effnet_w2v2)

Runs a clean Step-22 sweep on `causal_multimodal_dataset_effnet_w2v2.csv` and compares against Step-9/Step-21.
This section uses `test_*` metric columns to avoid `KeyError` from legacy `acc/auc/f1` sort keys.


In [ ]:
from pathlib import Path
import subprocess
import json
from datetime import datetime
import pandas as pd

# Robust root/python resolution.
if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'scripts').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'scripts').exists() and (p / 'src').exists()), cands[-1])

if 'PYTHON' in globals():
    PYTHON = Path(PYTHON)
if 'PYTHON' not in globals() or not PYTHON.exists():
    py = BACKEND_ROOT / '.venv' / 'bin' / 'python'
    import sys
    PYTHON = py if py.exists() else Path(sys.executable)

RUN_STEP22_SPEC_SWEEP = False  # set True to execute full Step-22 sweep
STEP22_PREFIX = 'fakeav_mrdf5cv_step22_realguard_focus'
STEP22_CONFIGS = [
    {'phase2_scenario_focus': 'none', 'phase2_scenario_focus_weight': 1.0, 'hard_negative_weight': 10.0, 'causal_breach_loss_weight': 0.05},
    {'phase2_scenario_focus': 'video_only_fake', 'phase2_scenario_focus_weight': 1.75, 'hard_negative_weight': 12.0, 'causal_breach_loss_weight': 0.05},
    {'phase2_scenario_focus': 'both_fake', 'phase2_scenario_focus_weight': 1.75, 'hard_negative_weight': 12.0, 'causal_breach_loss_weight': 0.05},
]
STEP22_OUT_CSV = BACKEND_ROOT / 'models/experiment_logs' / f'comparison_nextsteps_{datetime.now().strftime("%Y%m%d_%H%M%S")}_step22_realguard_focus_effnet.csv'

COMMON = [
    '--processed-csv', 'data/processed/causal_multimodal_dataset_effnet_w2v2.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models',
    '--logs-dir', 'models/experiment_logs',
    '--seed', '1337',
    '--split-seed', '42',
    '--n-splits', '5',
    '--val-size', '0.15',
    '--feature-profile', 'extended',
    '--epochs', '35',
    '--patience', '8',
    '--batch-size', '128',
    '--lr', '0.0003',
    '--weight-decay', '0.0001',
    '--loss', 'focal',
    '--focal-alpha', '0.40',
    '--focal-gamma', '2.0',
    '--train-weight-application', 'both',
    '--enable-multitask',
    '--multitask-weight', '0.25',
    '--ranking-loss-weight', '0.20',
    '--ranking-margin', '0.20',
    '--ranking-max-pairs', '1024',
    '--enable-lip-stream',
    '--causal-breach-target', 'heuristic',
    '--pretrain-enable',
    '--pretrain-epochs', '12',
    '--pretrain-patience', '4',
    '--phase2-enable',
    '--phase2-rounds', '2',
    '--phase2-hardneg-source', 'train',
    '--phase2-hardpos-source', 'train',
    '--phase2-use-hard-positives',
    '--hard-positive-weight', '2.0',
    '--hardpos-scenario', 'audio_only_fake',
    '--eval-threshold-source', 'val_acc',
    '--eval-threshold-priority', 'accuracy',
    '--phase-selection-priority', 'accuracy',
]

if RUN_STEP22_SPEC_SWEEP:
    rows = []
    for cfg in STEP22_CONFIGS:
        focus = cfg['phase2_scenario_focus']
        fw = cfg['phase2_scenario_focus_weight']
        hn = cfg['hard_negative_weight']
        cb = cfg['causal_breach_loss_weight']

        run_prefix = f"{STEP22_PREFIX}-{focus}_w{str(fw).replace('.', 'p')}_hn{int(hn)}_cb{str(cb).replace('.', 'p')}"
        cmd = [
            str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
            '--run-prefix', run_prefix,
            '--phase2-scenario-focus', str(focus),
            '--phase2-scenario-focus-weight', str(float(fw)),
            '--hard-negative-weight', str(float(hn)),
            '--causal-breach-loss-weight', str(float(cb)),
            *COMMON,
        ]

        print('
RUN:', run_prefix)
        cp = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True)

        row = {
            'run_prefix': run_prefix,
            'phase2_scenario_focus': focus,
            'phase2_scenario_focus_weight': float(fw),
            'hard_negative_weight': float(hn),
            'causal_breach_loss_weight': float(cb),
            'return_code': int(cp.returncode),
        }

        cands = sorted((BACKEND_ROOT / 'models/experiment_logs').glob(f'{run_prefix}_s1337_*'))
        run_dir = cands[-1] if cands else None
        row['run_tag'] = run_dir.name if run_dir else None
        row['run_dir'] = str(run_dir) if run_dir else None

        if run_dir and (run_dir / 'cv_summary.json').exists():
            obj = json.loads((run_dir / 'cv_summary.json').read_text())
            cm = obj.get('cv_metrics', {})
            row['num_folds_successful'] = int(obj.get('num_folds_successful', 0))
            row['num_folds_total'] = int(obj.get('num_folds_total', 0))
            for k in [
                'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc',
                'test_bal_acc', 'test_f1_real',
                'test_auc_manip_audio_only', 'test_auc_manip_video_only', 'test_auc_manip_both_fake'
            ]:
                v = cm.get(k, {})
                row[k] = float(v.get('mean')) if isinstance(v, dict) and v.get('mean') is not None else None

        rows.append(row)
        pd.DataFrame(rows).to_csv(STEP22_OUT_CSV, index=False)

    print('saved:', STEP22_OUT_CSV)
else:
    print('RUN_STEP22_SPEC_SWEEP=False (skip)')
    latest = sorted((BACKEND_ROOT / 'models/experiment_logs').glob('comparison_nextsteps_*_step22_realguard_focus_effnet.csv'))
    if latest:
        print('latest existing:', latest[-1])


In [ ]:
import pandas as pd
import json
from pathlib import Path

logs_root = BACKEND_ROOT / 'models/experiment_logs'
step22_csvs = sorted(logs_root.glob('comparison_nextsteps_*_step22_realguard_focus_effnet.csv'))
step21_csvs = sorted(logs_root.glob('comparison_nextsteps_*_step21_spec_recovery.csv'))

if not step22_csvs:
    print('No Step-22 sweep CSV found.')
else:
    s22 = pd.read_csv(step22_csvs[-1])
    metric_cols = [
        'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real',
        'test_auc_manip_audio_only', 'test_auc_manip_video_only', 'test_auc_manip_both_fake',
        'num_folds_successful', 'num_folds_total'
    ]
    for c in metric_cols:
        if c in s22.columns:
            s22[c] = pd.to_numeric(s22[c], errors='coerce')

    rank_cols = [c for c in ['test_acc', 'test_auc', 'test_f1'] if c in s22.columns]
    if rank_cols:
        s22 = s22.sort_values(rank_cols, ascending=[False] * len(rank_cols)).reset_index(drop=True)

    display_cols = [
        c for c in [
            'run_prefix', 'run_tag', 'return_code',
            'num_folds_successful', 'num_folds_total',
            'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real',
            'test_auc_manip_audio_only', 'test_auc_manip_video_only', 'test_auc_manip_both_fake'
        ]
        if c in s22.columns
    ]
    display(s22[display_cols])

    best22 = s22.iloc[0] if not s22.empty else None

    baseline_path = logs_root / 'fakeav_mrdf5cv_step9_densegrid_wmboth_r2_hn8_multiseed_s1337_20260313_131255/cv_summary.json'
    best21 = None
    if step21_csvs:
        s21 = pd.read_csv(step21_csvs[-1])
        for c in ['test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real']:
            if c in s21.columns:
                s21[c] = pd.to_numeric(s21[c], errors='coerce')
        rank21 = [c for c in ['test_acc', 'test_auc', 'test_f1'] if c in s21.columns]
        if rank21 and not s21.empty:
            s21 = s21.sort_values(rank21, ascending=[False] * len(rank21)).reset_index(drop=True)
            best21 = s21.iloc[0]

    if baseline_path.exists() and best22 is not None:
        b = json.loads(baseline_path.read_text())
        bm = b.get('cv_metrics', {})
        bvals = {k: float((bm.get(k) or {}).get('mean')) for k in ['test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real']}

        rows = [
            {
                'family': 'step9_baseline',
                'run_tag': b.get('run_tag', 'step9'),
                **bvals,
            },
            {
                'family': 'step22_best_effnet',
                'run_tag': best22.get('run_tag'),
                **{k: float(best22.get(k, float('nan'))) for k in bvals.keys()},
            },
        ]
        if best21 is not None:
            rows.insert(
                1,
                {
                    'family': 'step21_best',
                    'run_tag': best21.get('run_tag'),
                    **{k: float(best21.get(k, float('nan'))) for k in bvals.keys()},
                },
            )

        comp = pd.DataFrame(rows)
        for k in bvals.keys():
            comp[f'delta_vs_step9_{k}'] = comp[k] - float(bvals[k])

        out_csv = logs_root / f'comparison_nextsteps_{Path(step22_csvs[-1]).stem.split("_")[2]}_step9_step21_step22_best_summary.csv'
        comp.to_csv(out_csv, index=False)

        print('
Best Step-22 candidate:', best22.get('run_tag', 'N/A'))
        for k in ['test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real']:
            v = float(best22.get(k, float('nan')))
            print(f'  {k}: {v:.6f} (delta vs Step-9 {v - bvals[k]:+.6f})')

        gate = (
            float(best22.get('test_acc', -1)) > bvals['test_acc']
            and float(best22.get('test_auc', -1)) > bvals['test_auc']
            and float(best22.get('test_rec', -1)) >= 0.88
        )
        print('
Promotion gate (acc↑ and auc↑ and rec>=0.88):', gate)
        if gate:
            print('Proceed to multiseed+ensemble on Step-22 best config.')
        else:
            print('Do not promote Step-22 results; keep Step-9 baseline.')

        print('
Saved family comparison summary to:', out_csv)
        print('Note: Step-9 and Step-22 use different feature-schema eras; compare directionally unless Step-9 is re-run on effnet_w2v2.')


## 23) Step-9 Effnet Control (Apples-to-Apples vs Step-22)

Runs the Step-9 recipe on `causal_multimodal_dataset_effnet_w2v2.csv` (multi-seed + ensemble)
and compares directly against Step-22 best run.


In [ ]:
from pathlib import Path
import subprocess
from datetime import datetime

# Robust root/python resolution.
if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'scripts').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'scripts').exists() and (p / 'src').exists()), cands[-1])

if 'PYTHON' in globals():
    PYTHON = Path(PYTHON)
if 'PYTHON' not in globals() or not PYTHON.exists():
    py = BACKEND_ROOT / '.venv' / 'bin' / 'python'
    import sys
    PYTHON = py if py.exists() else Path(sys.executable)

RUN_STEP9_EFFNET_CONTROL = False  # set True to execute full multi-seed control
STEP9_EFFNET_PREFIX = 'fakeav_mrdf5cv_step9_effnetctrl_wmboth_r2_hn8_multiseed'

cmd = [
    str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
    '--processed-csv', 'data/processed/causal_multimodal_dataset_effnet_w2v2.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models',
    '--logs-dir', 'models/experiment_logs',
    '--run-prefix', STEP9_EFFNET_PREFIX,
    '--seed-list', '42,1337,2026',
    '--split-seed', '42',
    '--n-splits', '5',
    '--val-size', '0.15',
    '--feature-profile', 'extended',
    '--epochs', '35',
    '--patience', '8',
    '--batch-size', '128',
    '--lr', '0.0003',
    '--weight-decay', '0.0001',
    '--loss', 'focal',
    '--focal-alpha', '0.4',
    '--focal-gamma', '2.0',
    '--train-weight-application', 'both',
    '--enable-multitask',
    '--multitask-weight', '0.25',
    '--ranking-loss-weight', '0.2',
    '--ranking-margin', '0.2',
    '--ranking-max-pairs', '1024',
    '--pretrain-enable',
    '--pretrain-epochs', '12',
    '--pretrain-patience', '4',
    '--phase2-enable',
    '--phase2-rounds', '2',
    '--phase2-hardneg-source', 'train',
    '--phase2-hardpos-source', 'train',
    '--hard-negative-weight', '8.0',
    '--hard-positive-weight', '2.5',
    '--hardpos-scenario', 'audio_only_fake',
    '--phase2-scenario-focus', 'none',
    '--phase2-scenario-focus-weight', '1.0',
    '--eval-threshold-source', 'val_acc',
    '--eval-threshold-priority', 'accuracy',
    '--phase-selection-priority', 'accuracy',
    '--ensemble-enable',
    '--ensemble-top-k', '0',
    '--ensemble-rank-metric', 'val_bal_acc',
    '--ensemble-weighting', 'uniform',
]

if RUN_STEP9_EFFNET_CONTROL:
    print('RUN:', ' '.join(cmd))
    cp = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True)
    if cp.returncode != 0:
        raise RuntimeError(f'Step-9 effnet control failed with code={cp.returncode}')
else:
    print('RUN_STEP9_EFFNET_CONTROL=False (skip)')
    latest = sorted((BACKEND_ROOT / 'models/experiment_logs').glob(f'{STEP9_EFFNET_PREFIX}_multiseed_*'))
    if latest:
        print('latest existing multiseed run:', latest[-1])


In [ ]:
import json
from pathlib import Path
import pandas as pd

logs_root = BACKEND_ROOT / 'models/experiment_logs'
step22_csvs = sorted(logs_root.glob('comparison_nextsteps_*_step22_realguard_focus_effnet.csv'))
ms_runs = sorted(logs_root.glob('fakeav_mrdf5cv_step9_effnetctrl_wmboth_r2_hn8_multiseed_multiseed_*/multiseed_summary.json'))

if not ms_runs:
    print('No Step-9 effnet control multiseed summary found.')
elif not step22_csvs:
    print('No Step-22 sweep CSV found.')
else:
    ms_path = ms_runs[-1]
    ms = json.loads(ms_path.read_text())

    def extract_from_cv(path: Path):
        d = json.loads(path.read_text())
        cv = d.get('cv_metrics', {})
        out = {}
        for k in [
            'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc',
            'test_bal_acc', 'test_f1_real', 'test_f1_fake',
            'test_auc_manip_audio_only', 'test_auc_manip_video_only', 'test_auc_manip_both_fake'
        ]:
            m = cv.get(k, {})
            out[k] = float(m.get('mean')) if isinstance(m, dict) and m.get('mean') is not None else float('nan')
        return out

    rows = []
    for item in ms.get('seed_runs', []):
        summary_path = Path(item.get('summary_path', ''))
        if summary_path.exists():
            rows.append({
                'family': 'step9_effnet_seed',
                'name': f"seed_{item.get('seed')}",
                'run_tag': item.get('run_tag'),
                **extract_from_cv(summary_path),
            })

    ens_path = Path((ms.get('ensemble') or {}).get('ensemble_summary_path', ''))
    if ens_path.exists():
        ens = json.loads(ens_path.read_text())
        cv = ens.get('cv_metrics', {})
        rows.append({
            'family': 'step9_effnet_ensemble',
            'name': 'ensemble',
            'run_tag': ens.get('run_tag', 'step9_effnet_ensemble'),
            **{k: float((cv.get(k) or {}).get('mean')) if (cv.get(k) or {}).get('mean') is not None else float('nan') for k in [
                'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc',
                'test_bal_acc', 'test_f1_real', 'test_f1_fake',
                'test_auc_manip_audio_only', 'test_auc_manip_video_only', 'test_auc_manip_both_fake'
            ]},
        })

    s22 = pd.read_csv(step22_csvs[-1])
    for c in s22.columns:
        if c.startswith('test_'):
            s22[c] = pd.to_numeric(s22[c], errors='coerce')
    best22 = s22.sort_values(['test_acc', 'test_auc', 'test_f1'], ascending=False).iloc[0]

    rows.append({
        'family': 'step22_best_effnet',
        'name': 'best_config',
        'run_tag': best22.get('run_tag'),
        **{k: float(best22.get(k, float('nan'))) for k in [
            'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc',
            'test_bal_acc', 'test_f1_real', 'test_f1_fake',
            'test_auc_manip_audio_only', 'test_auc_manip_video_only', 'test_auc_manip_both_fake'
        ]},
    })

    comp = pd.DataFrame(rows)
    base = comp[comp['family'].eq('step22_best_effnet')].iloc[0]
    for k in ['test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real']:
        comp[f'delta_vs_step22_{k}'] = comp[k] - float(base[k])

    out_csv = logs_root / 'comparison_nextsteps_20260314_step9_effnetctrl_vs_step22.csv'
    comp.to_csv(out_csv, index=False)

    display_cols = [
        'family', 'name', 'run_tag',
        'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real',
        'delta_vs_step22_test_acc', 'delta_vs_step22_test_auc', 'delta_vs_step22_test_rec'
    ]
    display(comp[display_cols].sort_values(['family', 'name']).reset_index(drop=True))

    print('Latest Step-9 effnet multiseed summary:', ms_path)
    print('Latest Step-22 sweep CSV:', step22_csvs[-1])
    print('Saved comparison CSV:', out_csv)


In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

# Robust root/python resolution.
if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'scripts').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'scripts').exists() and (p / 'src').exists()), cands[-1])

if 'PYTHON' in globals():
    PYTHON = Path(PYTHON)
if 'PYTHON' not in globals() or not PYTHON.exists():
    py = BACKEND_ROOT / '.venv' / 'bin' / 'python'
    import sys
    PYTHON = py if py.exists() else Path(sys.executable)

RUN_STEP24_PREPROCESS = False  # set True to execute
IN_CSV = BACKEND_ROOT / 'data/processed/causal_multimodal_dataset_effnet_w2v2.csv'
OUT_CSV = BACKEND_ROOT / 'data/processed/causal_multimodal_dataset_effnet_w2v2_physfix.csv'

if RUN_STEP24_PREPROCESS:
    cmd = [
        str(PYTHON), 'scripts/augment_processed_with_artifact_proxies.py',
        '--input-csv', str(IN_CSV.relative_to(BACKEND_ROOT)),
        '--output-csv', str(OUT_CSV.relative_to(BACKEND_ROOT)),
    ]
    print('RUN:', ' '.join(cmd))
    cp = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True)
    if cp.returncode != 0:
        raise RuntimeError(f'augment_processed_with_artifact_proxies.py failed: code={cp.returncode}')
else:
    print('RUN_STEP24_PREPROCESS=False (skip)')

if not OUT_CSV.exists():
    raise FileNotFoundError(OUT_CSV)

df = pd.read_csv(OUT_CSV, nrows=2)
need = [
    'video_motion_noise_ratio',
    'video_shape_noise_ratio',
    'video_temporal_instability',
    'video_detection_dropout',
    'video_compression_proxy',
]
missing = [c for c in need if c not in df.columns]
print('physfix csv:', OUT_CSV)
print('ncols:', len(df.columns))
print('missing artifact columns:', missing)


In [ ]:
from pathlib import Path
import subprocess
import json
from datetime import datetime
import pandas as pd

RUN_STEP24_SWEEP = False  # set True to execute full 8-run sweep
STEP24_PREFIX = 'fakeav_mrdf5cv_step24_pareto'
STEP24_OUT_CSV = BACKEND_ROOT / 'models/experiment_logs' / f'comparison_nextsteps_{datetime.now().strftime("%Y%m%d_%H%M%S")}_step24_pareto_single_seed.csv'

CFGS = [
    {'name': 'a_step22_base',    'lip': True,  'cbw': 0.05, 'hnw': 10.0, 'hardpos': True},
    {'name': 'a_step22_cbw002',  'lip': True,  'cbw': 0.02, 'hnw': 10.0, 'hardpos': True},
    {'name': 'a_step22_hn8',     'lip': True,  'cbw': 0.05, 'hnw': 8.0,  'hardpos': True},
    {'name': 'a_step22_lipoff',  'lip': False, 'cbw': 0.05, 'hnw': 10.0, 'hardpos': True},
    {'name': 'b_step9_base',     'lip': False, 'cbw': 0.00, 'hnw': 8.0,  'hardpos': False},
    {'name': 'b_step9_hn10',     'lip': False, 'cbw': 0.00, 'hnw': 10.0, 'hardpos': False},
    {'name': 'b_step9_lipon',    'lip': True,  'cbw': 0.00, 'hnw': 8.0,  'hardpos': False},
    {'name': 'b_step9_cbw002',   'lip': False, 'cbw': 0.02, 'hnw': 8.0,  'hardpos': False},
]

COMMON = [
    '--processed-csv', 'data/processed/causal_multimodal_dataset_effnet_w2v2_physfix.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models',
    '--logs-dir', 'models/experiment_logs',
    '--seed', '1337',
    '--split-seed', '42',
    '--n-splits', '5',
    '--val-size', '0.15',
    '--feature-profile', 'extended',
    '--epochs', '35',
    '--patience', '8',
    '--batch-size', '128',
    '--lr', '0.0003',
    '--weight-decay', '0.0001',
    '--loss', 'focal',
    '--focal-alpha', '0.40',
    '--focal-gamma', '2.0',
    '--train-weight-application', 'both',
    '--enable-multitask',
    '--multitask-weight', '0.25',
    '--ranking-loss-weight', '0.20',
    '--ranking-margin', '0.20',
    '--ranking-max-pairs', '1024',
    '--phase2-enable',
    '--phase2-rounds', '2',
    '--phase2-hardneg-source', 'train',
    '--phase2-hardpos-source', 'train',
    '--hard-positive-weight', '2.0',
    '--hardpos-scenario', 'audio_only_fake',
    '--phase2-scenario-focus', 'none',
    '--phase2-scenario-focus-weight', '1.0',
    '--eval-threshold-source', 'val_acc',
    '--eval-threshold-priority', 'accuracy',
    '--phase-selection-priority', 'accuracy',
    '--pretrain-enable',
    '--pretrain-epochs', '12',
    '--pretrain-patience', '4',
]

if RUN_STEP24_SWEEP:
    rows = []
    for cfg in CFGS:
        run_prefix = f"{STEP24_PREFIX}_{cfg['name']}"
        cmd = [
            str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
            '--run-prefix', run_prefix,
            '--hard-negative-weight', str(float(cfg['hnw'])),
            '--causal-breach-loss-weight', str(float(cfg['cbw'])),
            '--causal-breach-target', 'heuristic' if float(cfg['cbw']) > 0 else 'none',
            *COMMON,
        ]
        if bool(cfg['lip']):
            cmd.append('--enable-lip-stream')
        if bool(cfg['hardpos']):
            cmd.append('--phase2-use-hard-positives')

        print()
        print('RUN:', run_prefix)
        cp = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True)

        row = {
            'config': cfg['name'],
            'run_prefix': run_prefix,
            'enable_lip_stream': bool(cfg['lip']),
            'causal_breach_loss_weight': float(cfg['cbw']),
            'hard_negative_weight': float(cfg['hnw']),
            'phase2_use_hard_positives': bool(cfg['hardpos']),
            'return_code': int(cp.returncode),
        }

        cands = sorted((BACKEND_ROOT / 'models/experiment_logs').glob(f'{run_prefix}_s1337_*'))
        run_dir = cands[-1] if cands else None
        row['run_tag'] = run_dir.name if run_dir else None

        if run_dir and (run_dir / 'cv_summary.json').exists():
            obj = json.loads((run_dir / 'cv_summary.json').read_text())
            cm = obj.get('cv_metrics', {}) or {}
            row['num_folds_successful'] = int(obj.get('num_folds_successful', 0))
            row['num_folds_total'] = int(obj.get('num_folds_total', 0))
            for k in ['test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real']:
                v = cm.get(k, {})
                row[k] = float(v.get('mean')) if isinstance(v, dict) and v.get('mean') is not None else None

        rows.append(row)
        pd.DataFrame(rows).to_csv(STEP24_OUT_CSV, index=False)

    print('saved:', STEP24_OUT_CSV)
else:
    print('RUN_STEP24_SWEEP=False (skip)')
    latest = sorted((BACKEND_ROOT / 'models/experiment_logs').glob('comparison_nextsteps_*_step24_pareto_single_seed.csv'))
    if latest:
        print('latest existing:', latest[-1])


In [ ]:
import pandas as pd
from pathlib import Path

logs_root = BACKEND_ROOT / 'models/experiment_logs'
sweep_csvs = sorted(logs_root.glob('comparison_nextsteps_*_step24_pareto_single_seed.csv'))
if not sweep_csvs:
    print('No Step-24 sweep CSV found.')
else:
    s = pd.read_csv(sweep_csvs[-1])
    for c in ['test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real']:
        if c in s.columns:
            s[c] = pd.to_numeric(s[c], errors='coerce')

    s['recall_ok_090'] = s['test_rec'] >= 0.90
    s = s.sort_values(['recall_ok_090', 'test_auc', 'test_acc', 'test_f1'], ascending=[False, False, False, False]).reset_index(drop=True)

    display_cols = [
        'config', 'run_tag', 'enable_lip_stream', 'causal_breach_loss_weight', 'hard_negative_weight', 'phase2_use_hard_positives',
        'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real', 'recall_ok_090'
    ]
    display(s[display_cols])

    best_recall_constrained = s[s['recall_ok_090']].head(1)
    best_unconstrained = s.head(1)
    if not best_recall_constrained.empty:
        r = best_recall_constrained.iloc[0]
        print('Best (recall>=0.90):', r['config'], r['run_tag'])
        print('  acc=', float(r['test_acc']), 'auc=', float(r['test_auc']), 'rec=', float(r['test_rec']))
    if not best_unconstrained.empty:
        u = best_unconstrained.iloc[0]
        print('Best (unconstrained):', u['config'], u['run_tag'])
        print('  acc=', float(u['test_acc']), 'auc=', float(u['test_auc']), 'rec=', float(u['test_rec']))


## 25) Step-25 Next Steps (Multiseed Branching + Threshold Tuning)

This section captures the next-step runs executed after Step-24:

- Branch A: `a_step22_lipoff` as multi-seed + ensemble.
- Branch B: `b_step9_base` as multi-seed + ensemble.
- Post-hoc threshold tuning on Branch A (recall-constrained policy).

Set `RUN_STEP25_MULTI=True` only when you want to rerun training (it is expensive).


In [ ]:
from pathlib import Path
import subprocess
import sys

# Robust root/python resolution.
if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'scripts').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'scripts').exists() and (p / 'src').exists()), cands[-1])

if 'PYTHON' in globals():
    PYTHON = Path(PYTHON)
if 'PYTHON' not in globals() or not PYTHON.exists():
    py = BACKEND_ROOT / '.venv' / 'bin' / 'python'
    PYTHON = py if py.exists() else Path(sys.executable)

RUN_STEP25_MULTI = False  # set True to execute both multiseed branches

STEP25_CFGS = [
    {
        'name': 'a_step22_lipoff',
        'run_prefix': 'fakeav_mrdf5cv_step25_lipoff_multiseed',
        'enable_lip_stream': False,
        'causal_breach_loss_weight': 0.05,
        'hard_negative_weight': 10.0,
        'phase2_use_hard_positives': True,
    },
    {
        'name': 'b_step9_base',
        'run_prefix': 'fakeav_mrdf5cv_step25_step9base_multiseed',
        'enable_lip_stream': False,
        'causal_breach_loss_weight': 0.0,
        'hard_negative_weight': 8.0,
        'phase2_use_hard_positives': False,
    },
]

COMMON = [
    '--processed-csv', 'data/processed/causal_multimodal_dataset_effnet_w2v2_physfix.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models',
    '--logs-dir', 'models/experiment_logs',
    '--seed', '1337',
    '--seed-list', '42,1337,2026',
    '--split-seed', '42',
    '--n-splits', '5',
    '--val-size', '0.15',
    '--feature-profile', 'extended',
    '--epochs', '35',
    '--patience', '8',
    '--batch-size', '128',
    '--lr', '0.0003',
    '--weight-decay', '0.0001',
    '--loss', 'focal',
    '--focal-alpha', '0.40',
    '--focal-gamma', '2.0',
    '--train-weight-application', 'both',
    '--enable-multitask',
    '--multitask-weight', '0.25',
    '--ranking-loss-weight', '0.20',
    '--ranking-margin', '0.20',
    '--ranking-max-pairs', '1024',
    '--phase2-enable',
    '--phase2-rounds', '2',
    '--phase2-hardneg-source', 'train',
    '--phase2-hardpos-source', 'train',
    '--hard-positive-weight', '2.0',
    '--hardpos-scenario', 'audio_only_fake',
    '--phase2-scenario-focus', 'none',
    '--phase2-scenario-focus-weight', '1.0',
    '--eval-threshold-source', 'val_acc',
    '--eval-threshold-priority', 'accuracy',
    '--phase-selection-priority', 'accuracy',
    '--ensemble-enable',
    '--ensemble-top-k', '3',
    '--ensemble-rank-metric', 'val_auc',
    '--ensemble-weighting', 'uniform',
]

if RUN_STEP25_MULTI:
    for cfg in STEP25_CFGS:
        cmd = [
            str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
            '--run-prefix', cfg['run_prefix'],
            '--causal-breach-loss-weight', str(float(cfg['causal_breach_loss_weight'])),
            '--hard-negative-weight', str(float(cfg['hard_negative_weight'])),
        ] + COMMON
        if cfg['enable_lip_stream']:
            cmd.append('--enable-lip-stream')
        if cfg['phase2_use_hard_positives']:
            cmd.append('--phase2-use-hard-positives')

        print('\nCMD:', ' '.join(cmd))
        res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
        if res.stdout:
            print(res.stdout[-5000:])
        if res.returncode != 0:
            if res.stderr:
                print(res.stderr[-5000:])
            raise RuntimeError(f"{cfg['name']} failed with code={res.returncode}")
else:
    print('RUN_STEP25_MULTI=False (skip training rerun).')

logs_root = BACKEND_ROOT / 'models/experiment_logs'
for cfg in STEP25_CFGS:
    ms = sorted(logs_root.glob(f"{cfg['run_prefix']}_multiseed_*/multiseed_summary.json"))
    ens = sorted(logs_root.glob(f"{cfg['run_prefix']}_multiseed_*/ensemble_summary.json"))
    print(f"{cfg['name']}: multiseed={ms[-1] if ms else 'MISSING'}")
    print(f"{cfg['name']}: ensemble={ens[-1] if ens else 'MISSING'}")


In [ ]:
from pathlib import Path
import json
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

logs_root = BACKEND_ROOT / 'models/experiment_logs'

METRICS = [
    'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc',
    'test_bal_acc', 'test_f1_real', 'test_f1_fake',
    'test_auc_manip_audio_only', 'test_auc_manip_video_only', 'test_auc_manip_both_fake',
]


def extract_cv_means(payload: dict) -> dict:
    cv = payload.get('cv_metrics', {}) if isinstance(payload, dict) else {}
    out = {}
    for k in METRICS:
        v = cv.get(k, {})
        out[k] = float(v.get('mean')) if isinstance(v, dict) and v.get('mean') is not None else float('nan')
    return out


def rows_from_multiseed(prefix: str, family: str):
    paths = sorted(logs_root.glob(f'{prefix}_multiseed_*/multiseed_summary.json'))
    if not paths:
        return [], None
    ms_path = paths[-1]
    ms = json.loads(ms_path.read_text())
    rows = []

    for seed_item in ms.get('seed_runs', []):
        p = Path(seed_item.get('summary_path', ''))
        if p.exists():
            d = json.loads(p.read_text())
            rows.append({
                'family': family,
                'name': f"seed_{seed_item.get('seed')}",
                'run_tag': seed_item.get('run_tag'),
                **extract_cv_means(d),
            })

    ep = Path((ms.get('ensemble') or {}).get('ensemble_summary_path', ''))
    if ep.exists():
        ed = json.loads(ep.read_text())
        rows.append({
            'family': family,
            'name': 'ensemble',
            'run_tag': ed.get('run_tag', f'{family}_ensemble'),
            **extract_cv_means(ed),
        })

    return rows, ms_path

rows = []
meta = []
for prefix, family in [
    ('fakeav_mrdf5cv_step25_lipoff_multiseed', 'a_step22_lipoff'),
    ('fakeav_mrdf5cv_step25_step9base_multiseed', 'b_step9_base'),
]:
    r, p = rows_from_multiseed(prefix, family)
    rows.extend(r)
    meta.append((family, p))

for fam, p in meta:
    print(f'{fam}: {p if p else "MISSING"}')

if not rows:
    print('No Step-25 summaries found. Run previous cell with RUN_STEP25_MULTI=True.')
else:
    df = pd.DataFrame(rows)
    order = ['family', 'name', 'run_tag'] + METRICS
    display(df[order].sort_values(['family', 'name']).reset_index(drop=True))

    best_acc = df.sort_values('test_acc', ascending=False).iloc[0]
    best_auc = df.sort_values('test_auc', ascending=False).iloc[0]
    best_rec = df.sort_values('test_rec', ascending=False).iloc[0]
    print('Best accuracy:', best_acc['family'], best_acc['name'], float(best_acc['test_acc']))
    print('Best AUC:', best_auc['family'], best_auc['name'], float(best_auc['test_auc']))
    print('Best recall:', best_rec['family'], best_rec['name'], float(best_rec['test_rec']))


In [ ]:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

logs_root = BACKEND_ROOT / 'models/experiment_logs'
tune_csvs = sorted(logs_root.glob('comparison_nextsteps_*_step25_lipoff_threshold_tuning.csv'))

if not tune_csvs:
    print('No Step-25 threshold tuning CSV found.')
    print('Expected pattern: models/experiment_logs/comparison_nextsteps_*_step25_lipoff_threshold_tuning.csv')
else:
    p = tune_csvs[-1]
    t = pd.read_csv(p)
    print('Loaded:', p)

    # Normalize a common schema for quick review.
    rename_map = {
        'acc_base': 'acc_before',
        'acc_tuned': 'acc_after',
        'rec_base': 'rec_before',
        'rec_tuned': 'rec_after',
        'auc_base': 'auc_before',
        'auc_tuned': 'auc_after',
    }
    for old, new in rename_map.items():
        if old in t.columns and new not in t.columns:
            t[new] = t[old]

    if {'acc_before', 'acc_after'}.issubset(t.columns):
        t['delta_acc'] = pd.to_numeric(t['acc_after'], errors='coerce') - pd.to_numeric(t['acc_before'], errors='coerce')
    if {'rec_before', 'rec_after'}.issubset(t.columns):
        t['delta_rec'] = pd.to_numeric(t['rec_after'], errors='coerce') - pd.to_numeric(t['rec_before'], errors='coerce')

    cols = [c for c in [
        'name', 'policy', 'acc_before', 'acc_after', 'delta_acc',
        'rec_before', 'rec_after', 'delta_rec',
        'auc_before', 'auc_after',
    ] if c in t.columns]

    display(t[cols] if cols else t)

    if {'name', 'rec_after'}.issubset(t.columns):
        rr = t[['name', 'rec_after']].copy()
        rr['recall_ok_090'] = pd.to_numeric(rr['rec_after'], errors='coerce') >= 0.90
        display(rr)


## 26) Balanced-Accuracy Recovery (Threshold Policy on Step-25 Ensemble)

This section reuses existing Step-25 seed checkpoints and recomputes **ensemble** metrics under a threshold selected for validation balanced accuracy.

Policies evaluated:
- `val_bal_acc` (pure balanced-accuracy thresholding)
- `val_bal_acc_rec>=0.88`
- `val_bal_acc_rec>=0.90`

Use this to improve specificity/real-class performance without retraining.


In [ ]:
from pathlib import Path
import importlib.util
import json
import numpy as np
import pandas as pd
import re

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# Robust root resolution.
if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'scripts').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'scripts').exists() and (p / 'src').exists()), cands[-1])

logs_root = BACKEND_ROOT / 'models' / 'experiment_logs'
STEP26_OUT = logs_root / 'comparison_nextsteps_20260314_step25_lipoff_ensemble_valbalacc.csv'

RUN_STEP26_RECOMPUTE = False  # set True to recompute from checkpoints

if RUN_STEP26_RECOMPUTE:
    script_path = BACKEND_ROOT / 'scripts' / 'run_fakeav_mrdf_5fold_cv.py'
    spec = importlib.util.spec_from_file_location('run_fakeav_mrdf_5fold_cv', script_path)
    mod = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(mod)

    def best_threshold_val_bal_acc(y, probs, recall_floor=None):
        y = np.asarray(y).astype(int)
        probs = np.asarray(probs, dtype=float)
        grid = np.linspace(0.0, 1.0, 1001, dtype=float)

        best = None
        best_key = (-np.inf, -np.inf, -np.inf)
        best_thr = 0.5
        for t in grid:
            pred = (probs >= float(t)).astype(int)
            m = mod._binary_confusion(y, pred)
            if recall_floor is not None and float(m['rec']) < float(recall_floor):
                continue
            key = (float(m['bal_acc']), float(m['acc']), float(m['f1']))
            if key > best_key:
                best_key = key
                best_thr = float(t)
                best = m

        if best is None:
            for t in grid:
                pred = (probs >= float(t)).astype(int)
                m = mod._binary_confusion(y, pred)
                key = (float(m['bal_acc']), float(m['acc']), float(m['f1']))
                if key > best_key:
                    best_key = key
                    best_thr = float(t)
                    best = m

        return best_thr

    def fold_num(v):
        m = re.search(r'(\d+)$', str(v))
        return int(m.group(1)) if m else -1

    seed_dirs = [
        logs_root / 'fakeav_mrdf5cv_step25_lipoff_multiseed_s42_20260314_104919',
        logs_root / 'fakeav_mrdf5cv_step25_lipoff_multiseed_s1337_20260314_105018',
        logs_root / 'fakeav_mrdf5cv_step25_lipoff_multiseed_s2026_20260314_105115',
    ]

    seed_fold_rows = []
    for sd in seed_dirs:
        rows = json.loads((sd / 'fold_metrics.json').read_text())
        seed_fold_rows.append({fold_num(r['fold']): r for r in rows})

    common_folds = sorted(set(seed_fold_rows[0]).intersection(*[set(m) for m in seed_fold_rows]))

    base_ens = json.loads((logs_root / 'fakeav_mrdf5cv_step25_lipoff_multiseed_multiseed_20260314_105213' / 'ensemble_summary.json').read_text())
    base_cv = base_ens['cv_metrics']
    base = {
        'test_acc': float(base_cv['test_acc']['mean']),
        'test_rec': float(base_cv['test_rec']['mean']),
        'test_bal_acc': float(base_cv['test_bal_acc']['mean']),
        'test_f1_real': float(base_cv['test_f1_real']['mean']),
        'test_auc': float(base_cv['test_auc']['mean']),
    }

    rows = []
    for recall_floor in [None, 0.88, 0.90]:
        fold_rows = []
        for f in common_folds:
            val_probs = []
            test_probs = []
            val_raw_ref = test_raw_ref = None
            yv_ref = yt_ref = None
            dt_ref = None

            for smap in seed_fold_rows:
                r = smap[f]
                model_dir = Path(r['model_dir'])
                val_csv = Path(r['val_csv'])
                test_csv = Path(r['test_csv'])

                val_raw, yv, dv, pv, _ = mod._predict_probs(model_dir=model_dir, csv_path=val_csv, feature_profile='extended')
                test_raw, yt, dt, pt, _ = mod._predict_probs(model_dir=model_dir, csv_path=test_csv, feature_profile='extended')

                if val_raw_ref is None:
                    val_raw_ref = val_raw
                    test_raw_ref = test_raw
                    yv_ref, yt_ref = yv, yt
                    dt_ref = dt

                val_probs.append(np.asarray(pv, dtype=float))
                test_probs.append(np.asarray(pt, dtype=float))

            p_val = np.mean(np.stack(val_probs, axis=0), axis=0)
            p_test = np.mean(np.stack(test_probs, axis=0), axis=0)
            thr = best_threshold_val_bal_acc(yv_ref, p_val, recall_floor=recall_floor)
            tm = mod._metrics_from_probs(test_raw_ref, yt_ref, dt_ref, p_test, thr)

            fold_rows.append({
                'fold': f,
                'threshold': float(thr),
                'test_acc': float(tm['acc']),
                'test_rec': float(tm['rec']),
                'test_bal_acc': float(tm['bal_acc']),
                'test_f1_real': float(tm['f1_real']),
                'test_auc': float(tm['auc']),
            })

        fd = pd.DataFrame(fold_rows)
        m = fd[[c for c in fd.columns if c != 'fold']].mean(numeric_only=True)
        policy = 'val_bal_acc' if recall_floor is None else f'val_bal_acc_rec>={recall_floor}'
        row = {'policy': policy, **{k: float(v) for k, v in m.items()}}
        row['delta_acc'] = row['test_acc'] - base['test_acc']
        row['delta_rec'] = row['test_rec'] - base['test_rec']
        row['delta_bal_acc'] = row['test_bal_acc'] - base['test_bal_acc']
        row['delta_f1_real'] = row['test_f1_real'] - base['test_f1_real']
        row['delta_auc'] = row['test_auc'] - base['test_auc']
        rows.append(row)

    out = pd.DataFrame(rows)
    out.to_csv(STEP26_OUT, index=False)
    print('Saved:', STEP26_OUT)
else:
    print('RUN_STEP26_RECOMPUTE=False (loading existing CSV if available).')

if STEP26_OUT.exists():
    df = pd.read_csv(STEP26_OUT)
    show_cols = [
        'policy', 'test_acc', 'test_rec', 'test_bal_acc', 'test_f1_real', 'test_auc',
        'delta_acc', 'delta_rec', 'delta_bal_acc', 'delta_f1_real', 'delta_auc'
    ]
    display(df[show_cols])
else:
    print('Missing:', STEP26_OUT)


In [ ]:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

logs_root = BACKEND_ROOT / 'models' / 'experiment_logs'
step26_csv = logs_root / 'comparison_nextsteps_20260314_step25_lipoff_ensemble_valbalacc.csv'

if not step26_csv.exists():
    print('Run previous cell first or set RUN_STEP26_RECOMPUTE=True.')
else:
    d = pd.read_csv(step26_csv)
    d = d.sort_values(['delta_bal_acc', 'test_acc'], ascending=[False, False]).reset_index(drop=True)
    display(d)

    # Practical deployment pick: preserve high recall while lifting balanced accuracy.
    cand = d[d['policy'] == 'val_bal_acc_rec>=0.88']
    if cand.empty:
        cand = d.head(1)

    r = cand.iloc[0]
    print('Recommended policy:', r['policy'])
    print('  test_acc=', float(r['test_acc']))
    print('  test_rec=', float(r['test_rec']))
    print('  test_bal_acc=', float(r['test_bal_acc']))
    print('  test_f1_real=', float(r['test_f1_real']))
    print('  delta_bal_acc=', float(r['delta_bal_acc']))


## 27) Balanced-Accuracy Retrain (Corrected With Pretrain)

This section runs a corrected balanced-accuracy retrain sweep with **global pretrain enabled** (matching Step-25 protocol), then promotes the best config to multiseed.

Important correction:
- Earlier Step-26 trials without pretrain underperformed and are not comparable with Step-25.
- This section keeps pretrain on (`--pretrain-enable`) for fair comparison.


In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# Robust root/python resolution.
if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'scripts').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'scripts').exists() and (p / 'src').exists()), cands[-1])

PYTHON = BACKEND_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON.exists():
    import sys
    PYTHON = Path(sys.executable)

RUN_STEP27_SINGLE_SEED = False  # set True to execute 2-run corrected sweep

common = [
    '--processed-csv', 'data/processed/causal_multimodal_dataset_effnet_w2v2_physfix.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models',
    '--logs-dir', 'models/experiment_logs',
    '--seed', '1337',
    '--split-seed', '42',
    '--n-splits', '5',
    '--val-size', '0.15',
    '--feature-profile', 'extended',
    '--epochs', '35',
    '--patience', '8',
    '--batch-size', '128',
    '--lr', '0.0003',
    '--weight-decay', '0.0001',
    '--loss', 'focal',
    '--focal-alpha', '0.40',
    '--focal-gamma', '2.0',
    '--train-weight-application', 'both',
    '--enable-multitask',
    '--multitask-weight', '0.25',
    '--ranking-loss-weight', '0.20',
    '--ranking-margin', '0.20',
    '--ranking-max-pairs', '1024',
    '--causal-breach-loss-weight', '0.05',
    '--causal-breach-target', 'heuristic',
    '--phase2-enable',
    '--phase2-rounds', '2',
    '--phase2-hardneg-source', 'train',
    '--phase2-hardpos-source', 'train',
    '--phase2-use-hard-positives',
    '--hard-negative-weight', '10.0',
    '--hard-positive-weight', '2.0',
    '--hardpos-scenario', 'audio_only_fake',
    '--phase2-scenario-focus', 'none',
    '--phase2-scenario-focus-weight', '1.0',
    '--pretrain-enable',
    '--pretrain-epochs', '12',
    '--pretrain-patience', '4',
    '--pretrain-scenario-focus', 'none',
    '--pretrain-scenario-focus-weight', '1.0',
    '--ensemble-enable',
]

cfgs = [
    {
        'name': 'valacc_balphase',
        'run_prefix': 'fakeav_mrdf5cv_step26p_balphase_valacc',
        'eval_threshold_source': 'val_acc',
        'eval_threshold_priority': 'accuracy',
        'phase_selection_priority': 'balanced_acc',
    },
    {
        'name': 'valtarget_balphase',
        'run_prefix': 'fakeav_mrdf5cv_step26p_balphase_valtarget',
        'eval_threshold_source': 'val_target',
        'eval_threshold_priority': 'balanced_acc',
        'phase_selection_priority': 'balanced_acc',
    },
]

if RUN_STEP27_SINGLE_SEED:
    for cfg in cfgs:
        cmd = [
            str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
            '--run-prefix', cfg['run_prefix'],
            '--eval-threshold-source', cfg['eval_threshold_source'],
            '--eval-threshold-priority', cfg['eval_threshold_priority'],
            '--phase-selection-priority', cfg['phase_selection_priority'],
        ] + common
        print('CMD:', ' '.join(cmd))
        res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True)
        if res.returncode != 0:
            raise RuntimeError(f"{cfg['name']} failed with code={res.returncode}")
else:
    print('RUN_STEP27_SINGLE_SEED=False (skip).')

logs_root = BACKEND_ROOT / 'models/experiment_logs'
rows = []
for cfg in cfgs:
    paths = sorted(logs_root.glob(f"{cfg['run_prefix']}_s1337_*/cv_summary.json"))
    if not paths:
        continue
    d = json.loads(paths[-1].read_text())
    cv = d.get('cv_metrics', {})
    rows.append({
        'config': cfg['name'],
        'run_tag': d.get('run_tag', paths[-1].parent.name),
        'summary_path': str(paths[-1]),
        'test_acc': float(cv['test_acc']['mean']),
        'test_prec': float(cv['test_prec']['mean']),
        'test_rec': float(cv['test_rec']['mean']),
        'test_f1': float(cv['test_f1']['mean']),
        'test_auc': float(cv['test_auc']['mean']),
        'test_bal_acc': float(cv['test_bal_acc']['mean']),
        'test_f1_real': float(cv['test_f1_real']['mean']),
    })

if rows:
    s = pd.DataFrame(rows).sort_values(['test_bal_acc', 'test_acc'], ascending=False).reset_index(drop=True)
    display(s)
else:
    print('No Step-27 single-seed summaries found yet.')


In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

RUN_STEP27_MULTI = False  # set True to execute multiseed promotion for valacc_balphase

if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'scripts').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'scripts').exists() and (p / 'src').exists()), cands[-1])

PYTHON = BACKEND_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON.exists():
    import sys
    PYTHON = Path(sys.executable)

cmd = [
    str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
    '--processed-csv', 'data/processed/causal_multimodal_dataset_effnet_w2v2_physfix.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models',
    '--logs-dir', 'models/experiment_logs',
    '--run-prefix', 'fakeav_mrdf5cv_step26p_balphase_valacc_multiseed',
    '--seed', '1337', '--seed-list', '42,1337,2026', '--split-seed', '42',
    '--n-splits', '5', '--val-size', '0.15', '--feature-profile', 'extended',
    '--epochs', '35', '--patience', '8', '--batch-size', '128', '--lr', '0.0003', '--weight-decay', '0.0001',
    '--loss', 'focal', '--focal-alpha', '0.40', '--focal-gamma', '2.0', '--train-weight-application', 'both',
    '--enable-multitask', '--multitask-weight', '0.25', '--ranking-loss-weight', '0.20', '--ranking-margin', '0.20', '--ranking-max-pairs', '1024',
    '--causal-breach-loss-weight', '0.05', '--causal-breach-target', 'heuristic',
    '--phase2-enable', '--phase2-rounds', '2', '--phase2-hardneg-source', 'train', '--phase2-hardpos-source', 'train', '--phase2-use-hard-positives',
    '--hard-negative-weight', '10.0', '--hard-positive-weight', '2.0', '--hardpos-scenario', 'audio_only_fake',
    '--phase2-scenario-focus', 'none', '--phase2-scenario-focus-weight', '1.0',
    '--pretrain-enable', '--pretrain-epochs', '12', '--pretrain-patience', '4', '--pretrain-scenario-focus', 'none', '--pretrain-scenario-focus-weight', '1.0',
    '--eval-threshold-source', 'val_acc', '--eval-threshold-priority', 'accuracy', '--phase-selection-priority', 'balanced_acc',
    '--ensemble-enable', '--ensemble-top-k', '3', '--ensemble-rank-metric', 'val_bal_acc', '--ensemble-weighting', 'rank_metric',
]

if RUN_STEP27_MULTI:
    print('CMD:', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True)
    if res.returncode != 0:
        raise RuntimeError(f'step27 multiseed failed: code={res.returncode}')
else:
    print('RUN_STEP27_MULTI=False (skip).')

logs_root = BACKEND_ROOT / 'models/experiment_logs'
ms = sorted(logs_root.glob('fakeav_mrdf5cv_step26p_balphase_valacc_multiseed_multiseed_*/multiseed_summary.json'))
ens = sorted(logs_root.glob('fakeav_mrdf5cv_step26p_balphase_valacc_multiseed_multiseed_*/ensemble_summary.json'))

print('latest multiseed summary:', ms[-1] if ms else 'MISSING')
print('latest ensemble summary:', ens[-1] if ens else 'MISSING')

if ens:
    d = json.loads(ens[-1].read_text())
    cv = d.get('cv_metrics', {})
    row = {
        'run_tag': d.get('run_tag', ens[-1].parent.name),
        'test_acc': float(cv['test_acc']['mean']),
        'test_prec': float(cv['test_prec']['mean']),
        'test_rec': float(cv['test_rec']['mean']),
        'test_f1': float(cv['test_f1']['mean']),
        'test_auc': float(cv['test_auc']['mean']),
        'test_bal_acc': float(cv['test_bal_acc']['mean']),
        'test_f1_real': float(cv['test_f1_real']['mean']),
    }
    display(pd.DataFrame([row]))


In [ ]:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'models').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'models').exists()), cands[-1])

csv_path = BACKEND_ROOT / 'models/experiment_logs/comparison_nextsteps_20260314_step27_balacc_progress.csv'
if not csv_path.exists():
    print('Missing:', csv_path)
else:
    d = pd.read_csv(csv_path)
    d = d.sort_values(['test_bal_acc', 'test_acc'], ascending=[False, False]).reset_index(drop=True)
    display(d)

    constrained = d[d['label'].str.contains('rec>=0.88', na=False)]
    if not constrained.empty:
        best = constrained.iloc[0]
        print('Best recall-constrained (>=0.88) policy:', best['label'])
        print('  test_acc=', float(best['test_acc']))
        print('  test_rec=', float(best['test_rec']))
        print('  test_bal_acc=', float(best['test_bal_acc']))
        print('  test_f1_real=', float(best['test_f1_real']))


## 28) Weighted Ensemble Policy Search (Metric Recovery)

This section evaluates post-hoc weighted seed-ensemble policies (step size `0.05`) and threshold calibration policies on validation folds.

Goal:
- improve balanced accuracy and real-class F1 without retraining,
- while offering recall-constrained operating points.

Generated artifacts used below:
- `comparison_nextsteps_20260314_step28_weighted_ensemble_search.csv`
- `comparison_nextsteps_20260314_step28_weighted_ensemble_search_step26p.csv`
- `comparison_nextsteps_20260314_step28_weighted_ensemble_finefloors.csv`
- `comparison_nextsteps_20260314_step28_progress.csv`


In [ ]:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'models').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'models').exists()), cands[-1])

logs_root = BACKEND_ROOT / 'models' / 'experiment_logs'
progress_csv = logs_root / 'comparison_nextsteps_20260314_step28_progress.csv'

if not progress_csv.exists():
    print('Missing:', progress_csv)
else:
    d = pd.read_csv(progress_csv)
    d = d.sort_values(['test_bal_acc', 'test_acc'], ascending=[False, False]).reset_index(drop=True)
    display(d)


In [ ]:
from pathlib import Path
import pandas as pd
import json

if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'models').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'models').exists()), cands[-1])

logs_root = BACKEND_ROOT / 'models' / 'experiment_logs'
progress_csv = logs_root / 'comparison_nextsteps_20260314_step28_progress.csv'

d = pd.read_csv(progress_csv)

best_overall = d.sort_values(['test_bal_acc', 'test_acc'], ascending=[False, False]).iloc[0]
best_rec85 = d[d['test_rec'] >= 0.85].sort_values(['test_bal_acc', 'test_acc'], ascending=[False, False]).iloc[0]
best_rec88 = d[d['test_rec'] >= 0.88].sort_values(['test_bal_acc', 'test_acc'], ascending=[False, False]).iloc[0]

print('Best overall bal_acc policy:', best_overall['label'])
print('  acc=', float(best_overall['test_acc']), 'rec=', float(best_overall['test_rec']), 'bal_acc=', float(best_overall['test_bal_acc']), 'f1_real=', float(best_overall['test_f1_real']))

print('Best policy with recall >= 0.85:', best_rec85['label'])
print('  acc=', float(best_rec85['test_acc']), 'rec=', float(best_rec85['test_rec']), 'bal_acc=', float(best_rec85['test_bal_acc']), 'f1_real=', float(best_rec85['test_f1_real']))

print('Best policy with recall >= 0.88:', best_rec88['label'])
print('  acc=', float(best_rec88['test_acc']), 'rec=', float(best_rec88['test_rec']), 'bal_acc=', float(best_rec88['test_bal_acc']), 'f1_real=', float(best_rec88['test_f1_real']))

# Save a compact recommendation artifact.
artifact = {
    'created_from': str(progress_csv),
    'best_overall_bal_acc': best_overall.to_dict(),
    'best_recall_ge_085': best_rec85.to_dict(),
    'best_recall_ge_088': best_rec88.to_dict(),
}
out_json = logs_root / 'deployment_policy_step28_candidates.json'
out_json.write_text(json.dumps(artifact, indent=2))
print('saved:', out_json)


## 29) Full-Mode Stress Test (>90 Core Metrics)

This section uses the new `--split-mode full` option (no 4-way downsampling) to test whether core metrics can exceed 90%.

Observed outcome:
- `acc/prec/rec/f1` exceed 90% in full mode,
- `auc` remains around `0.81`, so **all-metrics>90** is still not reached with current model family.


In [ ]:
from pathlib import Path
import subprocess

# Robust root/python resolution.
if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'scripts').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'scripts').exists() and (p / 'src').exists()), cands[-1])

PYTHON = BACKEND_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON.exists():
    import sys
    PYTHON = Path(sys.executable)

RUN_STEP29_FULLMODE = True  # set True to execute

cmd = [
    str(PYTHON), 'scripts/run_fakeav_mrdf_5fold_cv.py',
    '--processed-csv', 'data/processed/causal_multimodal_dataset_effnet_w2v2_physfix.csv',
    '--out-root', 'data/processed/causal_multimodal_dataset_fakeav_mrdf5cv',
    '--models-dir', 'models',
    '--logs-dir', 'models/experiment_logs',
    '--run-prefix', 'fakeav_mrdf5cv_step29_fullmode_pilot',
    '--seed', '1337', '--split-seed', '42',
    '--n-splits', '5', '--max-folds', '3', '--val-size', '0.15',
    '--split-mode', 'full',
    '--feature-profile', 'extended',
    '--epochs', '20', '--patience', '5', '--batch-size', '128', '--lr', '0.0003', '--weight-decay', '0.0001',
    '--loss', 'focal', '--focal-alpha', '0.40', '--focal-gamma', '2.0',
    '--train-weight-application', 'none',
    '--enable-multitask', '--multitask-weight', '0.10', '--ranking-loss-weight', '0.10', '--ranking-margin', '0.20', '--ranking-max-pairs', '512',
    '--causal-breach-loss-weight', '0.0', '--causal-breach-target', 'none',
    '--phase2-enable', '--phase2-rounds', '1', '--phase2-hardneg-source', 'train', '--phase2-hardpos-source', 'train', '--phase2-use-hard-positives',
    '--hard-negative-weight', '2.0', '--hard-positive-weight', '1.0', '--hardpos-scenario', 'audio_only_fake',
    '--phase2-scenario-focus', 'none', '--phase2-scenario-focus-weight', '1.0',
    '--pretrain-enable', '--pretrain-epochs', '6', '--pretrain-patience', '3', '--pretrain-scenario-focus', 'none', '--pretrain-scenario-focus-weight', '1.0',
    '--eval-threshold-source', 'val_acc', '--eval-threshold-priority', 'accuracy', '--phase-selection-priority', 'accuracy',
]

if RUN_STEP29_FULLMODE:
    print('CMD:', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True)
    if res.returncode != 0:
        raise RuntimeError(f'step29 fullmode run failed: code={res.returncode}')
else:
    print('RUN_STEP29_FULLMODE=False (skip).')


In [ ]:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'models').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'models').exists()), cands[-1])

csv_path = BACKEND_ROOT / 'models/experiment_logs/comparison_nextsteps_20260314_step29_fullmode_summary.csv'
if not csv_path.exists():
    print('Missing:', csv_path)
else:
    d = pd.read_csv(csv_path)
    display(d)

    if {'target_acc90', 'target_prec90', 'target_rec90', 'target_f190', 'target_auc90'}.issubset(d.columns):
        d['all_targets_90'] = d[['target_acc90', 'target_prec90', 'target_rec90', 'target_f190', 'target_auc90']].all(axis=1)
        display(d[['label', 'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'all_targets_90']])


## 30) Metrics Dashboard + Change Log

This section provides a single metrics view against targets and documents key pipeline/training changes implemented during optimization.

### Implemented Changes (Chronological Summary)

- **Data preprocessing**
  - Trimmed first 100ms audio head to reduce silence-shortcut bias.
  - Regenerated/augmented processed CSV with artifact proxy features.

- **Feature upgrades**
  - Added `EfficientNet-B4` face embedding features.
  - Added `Wav2Vec2 Base` fine-tuned embedding features.
  - Added lip ROI / dedicated lip-stream feature pathway (ablated on/off by config).
  - Added video artifact proxy features (`video_motion_noise_ratio`, `video_shape_noise_ratio`, etc.).

- **Training protocol**
  - MRDF-style scenario-aware CV splits (uniform 4-way) with weighted sampler options.
  - Weighted loss options (`focal` / weighted BCE paths) and scenario-focused weighting.
  - Stage-wise training with optional global pretrain + phase-2 hard-example mining.
  - Added causal-breach auxiliary supervision (`causal_breach_loss_weight`, heuristic target).

- **Evaluation and thresholding**
  - Added robust threshold policies (`val_acc`, `val_target`, post-hoc `val_bal_acc` with recall floors).
  - Added multiseed training + ensemble aggregation with configurable ranking/weighting.
  - Added weighted-seed ensemble search for better balanced-accuracy operating points.

- **Protocol extension for stress-test**
  - Added `--split-mode full` to run full-data scenario-stratified CV (no uniform downsampling).

### Current conclusion

- Full-mode runs reached very high `acc/prec/rec/f1` (>0.97) but AUC remained around ~0.81.
- Therefore **all-metrics >90% is still not achieved** with current stack; AUC is the limiting metric.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# Robust root resolution.
if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'models').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'models').exists()), cands[-1])

logs_root = BACKEND_ROOT / 'models' / 'experiment_logs'

# Source files from recent sections.
step28_csv = logs_root / 'comparison_nextsteps_20260314_step28_progress.csv'
step29_csv = logs_root / 'comparison_nextsteps_20260314_step29_fullmode_summary.csv'

rows = []
if step28_csv.exists():
    s28 = pd.read_csv(step28_csv)
    # Keep a few notable policies.
    keep_labels = [
        'step25_policy_val_bal_acc_rec>=0.88',
        'step28_step25_weighted_balacc',
        'step28_step25_weighted_balacc_rec>=0.86',
        'step28_step25_weighted_balacc_rec>=0.9',
    ]
    s28 = s28[s28['label'].isin(keep_labels)].copy()
    for _, r in s28.iterrows():
        rows.append({
            'run_label': r['label'],
            'test_acc': float(r['test_acc']),
            'test_prec': float(r['test_prec']) if pd.notna(r.get('test_prec', np.nan)) else np.nan,
            'test_rec': float(r['test_rec']),
            'test_f1': float(r['test_f1']) if pd.notna(r.get('test_f1', np.nan)) else np.nan,
            'test_auc': float(r['test_auc']),
            'test_bal_acc': float(r['test_bal_acc']) if pd.notna(r.get('test_bal_acc', np.nan)) else np.nan,
            'test_f1_real': float(r['test_f1_real']) if pd.notna(r.get('test_f1_real', np.nan)) else np.nan,
            'source': str(step28_csv),
        })

if step29_csv.exists():
    s29 = pd.read_csv(step29_csv)
    for _, r in s29.iterrows():
        rows.append({
            'run_label': r['label'],
            'test_acc': float(r['test_acc']),
            'test_prec': float(r['test_prec']) if pd.notna(r.get('test_prec', np.nan)) else np.nan,
            'test_rec': float(r['test_rec']),
            'test_f1': float(r['test_f1']) if pd.notna(r.get('test_f1', np.nan)) else np.nan,
            'test_auc': float(r['test_auc']),
            'test_bal_acc': float(r['test_bal_acc']) if pd.notna(r.get('test_bal_acc', np.nan)) else np.nan,
            'test_f1_real': float(r['test_f1_real']) if pd.notna(r.get('test_f1_real', np.nan)) else np.nan,
            'source': str(step29_csv),
        })

if not rows:
    print('No summary CSVs found. Run Step-28/Step-29 cells first.')
else:
    d = pd.DataFrame(rows).drop_duplicates(subset=['run_label']).reset_index(drop=True)

    # Targets from earlier requirement + strict-all-90 view.
    targets_orig = {
        'test_acc': 0.90,
        'test_f1': 0.88,
        'test_auc': 0.91,
        'test_prec': 0.87,
        'test_rec': 0.88,
    }
    targets_90 = {
        'test_acc': 0.90,
        'test_f1': 0.90,
        'test_auc': 0.90,
        'test_prec': 0.90,
        'test_rec': 0.90,
    }

    for k, v in targets_orig.items():
        d[f'pass_orig_{k}'] = pd.to_numeric(d[k], errors='coerce') >= float(v)
    for k, v in targets_90.items():
        d[f'pass_90_{k}'] = pd.to_numeric(d[k], errors='coerce') >= float(v)

    d['pass_all_orig_targets'] = d[[f'pass_orig_{k}' for k in targets_orig]].all(axis=1)
    d['pass_all_90_targets'] = d[[f'pass_90_{k}' for k in targets_90]].all(axis=1)

    display_cols = [
        'run_label', 'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real',
        'pass_all_orig_targets', 'pass_all_90_targets'
    ]
    display(d[display_cols].sort_values(['pass_all_90_targets', 'pass_all_orig_targets', 'test_auc', 'test_acc'], ascending=[False, False, False, False]).reset_index(drop=True))

    # Explicit best-by-metric snapshot.
    metrics = ['test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc', 'test_bal_acc', 'test_f1_real']
    best_rows = []
    for m in metrics:
        tmp = d[['run_label', m]].copy()
        tmp[m] = pd.to_numeric(tmp[m], errors='coerce')
        tmp = tmp.dropna(subset=[m])
        if tmp.empty:
            continue
        idx = tmp[m].idxmax()
        best_rows.append({'metric': m, 'best_run': d.loc[idx, 'run_label'], 'value': float(d.loc[idx, m])})
    print('Best run per metric:')
    display(pd.DataFrame(best_rows))


### Confusion Metrics for Current Best Runs

Builds confusion metrics for the **current best saved runs** (best by `acc`, `f1`, `auc`, `bal_acc`) where fold-level confusion counts are available.

Computed metrics:
- `tp, tn, fp, fn`
- `accuracy, precision, recall, specificity, f1_fake`
- `precision_real, recall_real, f1_real`
- `balanced_acc`


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'models').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'models').exists()), cands[-1])

logs_root = BACKEND_ROOT / 'models' / 'experiment_logs'

def latest_one(pattern: str):
    paths = sorted(logs_root.glob(pattern))
    return paths[-1] if paths else None

# Candidate runs with summary + fold-level confusion artifacts.
candidates = []
def add_candidate(label, summary_path, fold_path):
    if summary_path and fold_path and summary_path.exists() and fold_path.exists():
        candidates.append({'label': label, 'summary': summary_path, 'fold': fold_path})

add_candidate(
    'step29_fullmode_pilot',
    latest_one('fakeav_mrdf5cv_step29_fullmode_pilot_s1337_*/cv_summary.json'),
    latest_one('fakeav_mrdf5cv_step29_fullmode_pilot_s1337_*/fold_metrics.json'),
)
add_candidate(
    'step29_fullmode_aucpush',
    latest_one('fakeav_mrdf5cv_step29_fullmode_aucpush_s1337_*/cv_summary.json'),
    latest_one('fakeav_mrdf5cv_step29_fullmode_aucpush_s1337_*/fold_metrics.json'),
)
add_candidate(
    'step25_ensemble_raw',
    latest_one('fakeav_mrdf5cv_step25_lipoff_multiseed_multiseed_*/ensemble_summary.json'),
    latest_one('fakeav_mrdf5cv_step25_lipoff_multiseed_multiseed_*/ensemble_fold_metrics.json'),
)
add_candidate(
    'step26p_ensemble_raw',
    latest_one('fakeav_mrdf5cv_step26p_balphase_valacc_multiseed_multiseed_*/ensemble_summary.json'),
    latest_one('fakeav_mrdf5cv_step26p_balphase_valacc_multiseed_multiseed_*/ensemble_fold_metrics.json'),
)

if not candidates:
    print('No candidate runs with fold confusion artifacts found.')
else:
    metric_rows = []
    for c in candidates:
        s = json.loads(c['summary'].read_text())
        cv = s.get('cv_metrics', {})
        metric_rows.append({
            'label': c['label'],
            'summary_path': str(c['summary']),
            'fold_path': str(c['fold']),
            'test_acc': float((cv.get('test_acc') or {}).get('mean', np.nan)),
            'test_f1': float((cv.get('test_f1') or {}).get('mean', np.nan)),
            'test_auc': float((cv.get('test_auc') or {}).get('mean', np.nan)),
            'test_bal_acc': float((cv.get('test_bal_acc') or {}).get('mean', np.nan)),
        })

    mdf = pd.DataFrame(metric_rows)
    chosen = set()
    for metric in ['test_acc', 'test_f1', 'test_auc', 'test_bal_acc']:
        tmp = mdf[['label', metric]].dropna(subset=[metric])
        if not tmp.empty:
            chosen.add(str(tmp.sort_values(metric, ascending=False).iloc[0]['label']))

    chosen_df = mdf[mdf['label'].isin(chosen)].copy().reset_index(drop=True)

    def agg_confusion(row):
        fold_data = json.loads(Path(row['fold_path']).read_text())
        if not isinstance(fold_data, list):
            fold_data = fold_data.get('folds', [])
        tp = float(sum(float(r.get('test_tp', 0.0)) for r in fold_data))
        tn = float(sum(float(r.get('test_tn', 0.0)) for r in fold_data))
        fp = float(sum(float(r.get('test_fp', 0.0)) for r in fold_data))
        fn = float(sum(float(r.get('test_fn', 0.0)) for r in fold_data))

        total = tp + tn + fp + fn + 1e-8
        acc = (tp + tn) / total
        prec = tp / (tp + fp + 1e-8)
        rec = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)
        f1_fake = 2.0 * prec * rec / (prec + rec + 1e-8)

        prec_real = tn / (tn + fn + 1e-8)
        rec_real = spec
        f1_real = 2.0 * prec_real * rec_real / (prec_real + rec_real + 1e-8)
        bal_acc = 0.5 * (rec + spec)

        return pd.Series({
            'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'specificity': spec,
            'f1_fake': f1_fake,
            'precision_real': prec_real,
            'recall_real': rec_real,
            'f1_real': f1_real,
            'balanced_acc': bal_acc,
        })

    conf = chosen_df.apply(agg_confusion, axis=1)
    out = pd.concat([chosen_df[['label', 'summary_path', 'fold_path']], conf], axis=1)

    print('Selected current-best runs (by acc/f1/auc/bal_acc among saved run summaries):')
    display(chosen_df[['label', 'test_acc', 'test_f1', 'test_auc', 'test_bal_acc']].sort_values('test_auc', ascending=False).reset_index(drop=True))

    print('Confusion metrics (aggregated across folds):')
    display(out[['label', 'tp', 'tn', 'fp', 'fn', 'accuracy', 'precision', 'recall', 'specificity', 'f1_fake', 'precision_real', 'recall_real', 'f1_real', 'balanced_acc']])

    out_csv = logs_root / 'comparison_nextsteps_20260314_step30_confusion_metrics.csv'
    out.to_csv(out_csv, index=False)
    print('saved:', out_csv)


### Confusion Matrix (2x2) for Current Best Runs

Matrix layout:
- rows = **actual** (`real`, `fake`)
- columns = **predicted** (`real`, `fake`)
- matrix format = `[[TN, FP], [FN, TP]]`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'models').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'models').exists()), cands[-1])

logs_root = BACKEND_ROOT / 'models' / 'experiment_logs'
conf_csv = logs_root / 'comparison_nextsteps_20260314_step30_confusion_metrics.csv'

if not conf_csv.exists():
    print('Missing confusion metrics CSV:', conf_csv)
    print('Run the previous "Confusion Metrics" cell first.')
else:
    d = pd.read_csv(conf_csv)
    req = ['label', 'tn', 'fp', 'fn', 'tp']
    missing = [c for c in req if c not in d.columns]
    if missing:
        raise RuntimeError(f'Missing required columns in {conf_csv}: {missing}')

    long_rows = []
    summary_rows = []

    for _, r in d.iterrows():
        label = str(r['label'])
        tn = int(round(float(r['tn'])))
        fp = int(round(float(r['fp'])))
        fn = int(round(float(r['fn'])))
        tp = int(round(float(r['tp'])))

        actual_real = tn + fp
        actual_fake = fn + tp
        pred_real = tn + fn
        pred_fake = fp + tp
        total = actual_real + actual_fake

        cm = np.array([[tn, fp], [fn, tp]], dtype=float)
        row_sum = cm.sum(axis=1, keepdims=True)
        col_sum = cm.sum(axis=0, keepdims=True)
        cm_row_norm = np.divide(cm, row_sum, out=np.zeros_like(cm), where=row_sum > 0)
        cm_col_norm = np.divide(cm, col_sum, out=np.zeros_like(cm), where=col_sum > 0)

        raw_df = pd.DataFrame(cm.astype(int), index=['actual_real', 'actual_fake'], columns=['pred_real', 'pred_fake'])
        row_df = pd.DataFrame(cm_row_norm, index=['actual_real', 'actual_fake'], columns=['pred_real', 'pred_fake'])
        col_df = pd.DataFrame(cm_col_norm, index=['actual_real', 'actual_fake'], columns=['pred_real', 'pred_fake'])

        counts_df = pd.DataFrame([
            {'item': 'TN (actual real -> pred real)', 'count': tn},
            {'item': 'FP (actual real -> pred fake)', 'count': fp},
            {'item': 'FN (actual fake -> pred real)', 'count': fn},
            {'item': 'TP (actual fake -> pred fake)', 'count': tp},
            {'item': 'Actual real total', 'count': actual_real},
            {'item': 'Actual fake total', 'count': actual_fake},
            {'item': 'Pred real total', 'count': pred_real},
            {'item': 'Pred fake total', 'count': pred_fake},
            {'item': 'Overall total', 'count': total},
        ])

        print(f'\nRun: {label}')
        print('Actual prediction numbers:')
        display(counts_df)
        print('Raw confusion matrix [[TN, FP], [FN, TP]]:')
        display(raw_df)
        print('Row-normalized (by actual class):')
        display(row_df)
        print('Column-normalized (by predicted class):')
        display(col_df)

        long_rows.extend([
            {'label': label, 'actual': 'real', 'pred': 'real', 'count': tn},
            {'label': label, 'actual': 'real', 'pred': 'fake', 'count': fp},
            {'label': label, 'actual': 'fake', 'pred': 'real', 'count': fn},
            {'label': label, 'actual': 'fake', 'pred': 'fake', 'count': tp},
        ])
        summary_rows.append({
            'label': label,
            'tn': tn,
            'fp': fp,
            'fn': fn,
            'tp': tp,
            'actual_real_total': actual_real,
            'actual_fake_total': actual_fake,
            'pred_real_total': pred_real,
            'pred_fake_total': pred_fake,
            'overall_total': total,
        })

    long_df = pd.DataFrame(long_rows)
    out_csv_long = logs_root / 'comparison_nextsteps_20260314_step30_confusion_matrices_long.csv'
    long_df.to_csv(out_csv_long, index=False)
    print('Saved long-format confusion matrices:', out_csv_long)

    out_csv_counts = logs_root / 'comparison_nextsteps_20260314_step30_confusion_counts.csv'
    pd.DataFrame(summary_rows).to_csv(out_csv_counts, index=False)
    print('Saved confusion count summary:', out_csv_counts)


## 31) Fix Zero-Real Predictions (Specificity-Constrained Thresholds)

Problem:
- In full-mode, `eval-threshold-source=val_acc` chooses a threshold that can predict almost all samples as fake.
- This yields high accuracy but near-zero real detection (`TN≈0`).

Fix:
- Select threshold on validation with constraints on minimum recall **and** minimum specificity.
- This forces non-zero real predictions and improves balanced behavior.


In [ ]:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

if 'BACKEND_ROOT' in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if 'BACKEND_ROOT' not in globals() or not (BACKEND_ROOT / 'models').exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'backend']
    BACKEND_ROOT = next((p for p in cands if (p / 'models').exists()), cands[-1])

logs_root = BACKEND_ROOT / 'models' / 'experiment_logs'
csv_path = logs_root / 'comparison_nextsteps_20260314_step30_fullmode_specfix_thresholds.csv'

if not csv_path.exists():
    print('Missing:', csv_path)
    print('Run threshold sweep script first (already executed in this run history).')
else:
    d = pd.read_csv(csv_path)
    d = d.sort_values(['bal_acc', 'acc'], ascending=[False, False]).reset_index(drop=True)
    display(d)

    # Recommended operating points by use-case.
    best_bal = d.iloc[0]
    best_high_acc = d[d['acc'] >= 0.95].sort_values(['spec', 'bal_acc'], ascending=[False, False]).head(1)

    print('Best balanced policy:')
    print('  policy=', best_bal['policy'])
    print('  acc=', float(best_bal['acc']), 'rec=', float(best_bal['rec']), 'spec=', float(best_bal['spec']), 'bal_acc=', float(best_bal['bal_acc']))
    print('  confusion(avg): TP=', float(best_bal['tp']), 'TN=', float(best_bal['tn']), 'FP=', float(best_bal['fp']), 'FN=', float(best_bal['fn']))

    if not best_high_acc.empty:
        r = best_high_acc.iloc[0]
        print('High-accuracy non-zero-real policy (acc>=0.95):')
        print('  policy=', r['policy'])
        print('  acc=', float(r['acc']), 'rec=', float(r['rec']), 'spec=', float(r['spec']), 'bal_acc=', float(r['bal_acc']))
        print('  confusion(avg): TP=', float(r['tp']), 'TN=', float(r['tn']), 'FP=', float(r['fp']), 'FN=', float(r['fn']))


## 32) Enforce Non-Zero Real-Class Predictions (Runner-Level Targets + Domain Guards)

Use runner-level target thresholds and domain constraints to avoid the high-accuracy / zero-real-prediction collapse.

Controls:
- `--target-spec`: minimum specificity target during target-threshold calibration.
- `--min-domain-spec` and `--min-domain-rec`: checkpoint constraints on worst-domain performance.
- `--train-target-priority balanced_acc`: stable tie-break rule for threshold calibration.


In [ ]:
# Step-32 execution cell (set RUN_STEP32_SPEC_GUARD=True to run).
from pathlib import Path
import subprocess, shlex, sys

def _resolve_backend_root() -> Path:
    cwd = Path.cwd().resolve()
    for cand in (cwd, cwd / "backend", cwd.parent):
        if (cand / "scripts" / "run_fakeav_mrdf_5fold_cv.py").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError(f"Could not resolve backend root from cwd={cwd}")

BACKEND_ROOT = _resolve_backend_root()
PYTHON = BACKEND_ROOT / ".venv" / "bin" / "python"
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

RUN_STEP32_SPEC_GUARD = True
STEP32_RUN_PREFIX = "fakeav_mrdf5cv_step32_specguard"
STEP32_TARGET_SPEC = 0.35
STEP32_MIN_DOMAIN_SPEC = 0.25
STEP32_MIN_DOMAIN_REC = 0.82

cmd = [
    str(PYTHON), "scripts/run_fakeav_mrdf_5fold_cv.py",
    "--processed-csv", "data/processed/causal_multimodal_dataset_effnet_w2v2_physfix.csv",
    "--out-root", "data/processed/causal_multimodal_dataset_fakeav_mrdf5cv",
    "--models-dir", "models",
    "--logs-dir", "models/experiment_logs",
    "--run-prefix", STEP32_RUN_PREFIX,
    "--split-mode", "full",
    "--n-splits", "5",
    "--feature-profile", "extended",
    "--loss", "focal",
    "--train-weight-application", "both",
    "--eval-threshold-source", "val_target",
    "--eval-threshold-priority", "balanced_acc",
    "--train-target-priority", "balanced_acc",
    "--target-acc", "0.90",
    "--target-precision", "0.87",
    "--target-recall", "0.88",
    "--target-f1", "0.88",
    "--target-spec", f"{float(STEP32_TARGET_SPEC):.4f}",
    "--min-domain-spec", f"{float(STEP32_MIN_DOMAIN_SPEC):.4f}",
    "--min-domain-rec", f"{float(STEP32_MIN_DOMAIN_REC):.4f}",
    "--phase2-enable",
    "--phase2-rounds", "3",
    "--phase2-hardneg-source", "val",
    "--hard-negative-weight", "6.0",
    "--phase2-use-hard-positives",
    "--phase2-hardpos-source", "val",
    "--hard-positive-weight", "3.0",
    "--hardpos-scenario", "none"
]

print("BACKEND_ROOT:", BACKEND_ROOT)
print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

if RUN_STEP32_SPEC_GUARD:
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f"run_fakeav_mrdf_5fold_cv.py failed: code={res.returncode}")
else:
    print("RUN_STEP32_SPEC_GUARD=False (preview only).")


## 33) Full Fake-Row Visual Refresh (Proxy) + Retrain

This section documents the leakage-mitigation follow-up after Step-32:
- `Step-33`: real-only visual refresh (true embeddings) showed inflated metrics due to missing fake visuals in prior CSV.
- `Step-34`: full fake-row visual refresh (proxy-mapped visual features) removes zero/non-zero class shortcut, then reruns 5-fold CV with Step-32 guard settings.

Use these cells to reproduce the full refresh + retrain path when needed.


In [ ]:
from pathlib import Path
import subprocess, shlex, sys

def _resolve_backend_root() -> Path:
    cwd = Path.cwd().resolve()
    for cand in (cwd, cwd / "backend", cwd.parent):
        if (cand / "scripts" / "run_fakeav_mrdf_5fold_cv.py").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError(f"Could not resolve backend root from cwd={cwd}")

BACKEND_ROOT = _resolve_backend_root()
PYTHON = BACKEND_ROOT / ".venv" / "bin" / "python"
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

# Safety flags: set True only when you want to run heavy jobs.
RUN_STEP34_REFRESH_PROXY = False
RUN_STEP34_CV = False

SRC_CSV = BACKEND_ROOT / "data/processed/causal_multimodal_dataset_effnet_w2v2_physfix_realvis.csv"
OUT_CSV = BACKEND_ROOT / "data/processed/causal_multimodal_dataset_effnet_w2v2_physfix_fullvis_proxy.csv"

refresh_cmd = [
    str(PYTHON), "scripts/refresh_fake_visual_proxy.py",
    "--input-csv", str(SRC_CSV.relative_to(BACKEND_ROOT)),
    "--output-csv", str(OUT_CSV.relative_to(BACKEND_ROOT)),
    "--workers", "8",
    "--chunksize", "256",
    "--sample-frames", "2",
    "--frame-stride", "24",
    "--frame-source", "first",
]

cv_cmd = [
    str(PYTHON), "scripts/run_fakeav_mrdf_5fold_cv.py",
    "--processed-csv", str(OUT_CSV.relative_to(BACKEND_ROOT)),
    "--out-root", "data/processed/causal_multimodal_dataset_fakeav_mrdf5cv",
    "--models-dir", "models",
    "--logs-dir", "models/experiment_logs",
    "--run-prefix", "fakeav_mrdf5cv_step34_fullvis_proxy",
    "--split-mode", "full",
    "--n-splits", "5",
    "--feature-profile", "extended",
    "--loss", "focal",
    "--train-weight-application", "both",
    "--eval-threshold-source", "val_target",
    "--eval-threshold-priority", "balanced_acc",
    "--train-target-priority", "balanced_acc",
    "--target-acc", "0.90",
    "--target-precision", "0.87",
    "--target-recall", "0.88",
    "--target-f1", "0.88",
    "--target-spec", "0.3500",
    "--min-domain-spec", "0.2500",
    "--min-domain-rec", "0.8200",
    "--phase2-enable",
    "--phase2-rounds", "3",
    "--phase2-hardneg-source", "val",
    "--hard-negative-weight", "6.0",
    "--phase2-use-hard-positives",
    "--phase2-hardpos-source", "val",
    "--hard-positive-weight", "3.0",
    "--hardpos-scenario", "none",
]

print("BACKEND_ROOT:", BACKEND_ROOT)
print("Refresh command:")
print(" ".join(shlex.quote(x) for x in refresh_cmd))
print("CV command:")
print(" ".join(shlex.quote(x) for x in cv_cmd))

if RUN_STEP34_REFRESH_PROXY:
    res = subprocess.run(refresh_cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f"refresh_fake_visual_proxy.py failed: code={res.returncode}")
else:
    print("RUN_STEP34_REFRESH_PROXY=False (skip)")

if RUN_STEP34_CV:
    res = subprocess.run(cv_cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f"run_fakeav_mrdf_5fold_cv.py failed: code={res.returncode}")
else:
    print("RUN_STEP34_CV=False (skip)")


## 34) Step-32/33/34 Metrics + Confusion Comparison

This cell compares the latest three milestone runs and prints actual confusion counts (`TN/FP/FN/TP`) so real-class performance is explicit.


In [ ]:
from pathlib import Path
import json
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

if "BACKEND_ROOT" in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if "BACKEND_ROOT" not in globals() or not (BACKEND_ROOT / "models").exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / "backend"]
    BACKEND_ROOT = next((p for p in cands if (p / "models").exists()), cands[-1])

logs_root = BACKEND_ROOT / "models" / "experiment_logs"

runs = {
    "step32_specguard": "fakeav_mrdf5cv_step32_specguard_s42_20260314_141209",
    "step33_realvis_leaky": "fakeav_mrdf5cv_step33_realvis_s42_20260314_150104",
    "step34_fullvis_proxy": "fakeav_mrdf5cv_step34_fullvis_proxy_s42_20260314_161749",
}

rows = []
for label, run_tag in runs.items():
    fold_csv = logs_root / run_tag / "fold_metrics.csv"
    summary_json = logs_root / run_tag / "cv_summary.json"
    if not fold_csv.exists() or not summary_json.exists():
        print(f"Missing artifacts for {label}: fold_csv={fold_csv.exists()} summary={summary_json.exists()}")
        continue

    d = pd.read_csv(fold_csv)
    if "error" in d.columns:
        d = d[d["error"].isna() | (d["error"].astype(str).str.strip() == "")].copy()

    tn = int(d["test_tn"].sum())
    fp = int(d["test_fp"].sum())
    fn = int(d["test_fn"].sum())
    tp = int(d["test_tp"].sum())

    rows.append({
        "label": label,
        "run_tag": run_tag,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "spec_real_recall": float(tn / (tn + fp)) if (tn + fp) else None,
        "rec_fake": float(tp / (tp + fn)) if (tp + fn) else None,
        "acc": float(d["test_acc"].mean()),
        "prec": float(d["test_prec"].mean()),
        "rec": float(d["test_rec"].mean()),
        "f1": float(d["test_f1"].mean()),
        "bal_acc": float(d["test_bal_acc"].mean()),
        "auc": float(d["test_auc"].mean()),
        "f1_real": float(d["test_f1_real"].mean()),
        "f1_fake": float(d["test_f1_fake"].mean()),
        "auc_audio_only": float(d["test_auc_manip_audio_only"].mean()),
        "auc_video_only": float(d["test_auc_manip_video_only"].mean()),
        "auc_both_fake": float(d["test_auc_manip_both_fake"].mean()),
    })

if not rows:
    print("No comparison rows built.")
else:
    out = pd.DataFrame(rows)
    display(out)
    out_csv = logs_root / "comparison_step32_step33_step34.csv"
    out.to_csv(out_csv, index=False)
    print("Saved:", out_csv)


## 35) Next Options Execution (Threshold Sweep + Step-35 Retrain)

This section captures the two requested next actions:
1. **Option-1**: threshold-only sweep on Step-34 checkpoints (no retrain).
2. **Option-2**: Step-35 retraining with stronger real-class guard and class-balanced objective.

Artifacts saved under `models/experiment_logs/`:
- `comparison_nextsteps_20260314_step34_threshold_sweep.csv`
- `comparison_nextsteps_20260314_step34_threshold_sweep_folds.csv`
- `fakeav_mrdf5cv_step35_realguard_*`
- `comparison_step34_step35_and_policy.csv`


In [ ]:
# Option-1: threshold-only sweep on Step-34 checkpoints.
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd

if "BACKEND_ROOT" in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if "BACKEND_ROOT" not in globals() or not (BACKEND_ROOT / "models").exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / "backend"]
    BACKEND_ROOT = next((p for p in cands if (p / "models").exists()), cands[-1])

logs = BACKEND_ROOT / "models" / "experiment_logs"
run_tag = "fakeav_mrdf5cv_step34_fullvis_proxy_s42_20260314_161749"
fold_csv = logs / run_tag / "fold_metrics.csv"
assert fold_csv.exists(), f"Missing: {fold_csv}"

spec = importlib.util.spec_from_file_location("mrdf", BACKEND_ROOT / "scripts" / "run_fakeav_mrdf_5fold_cv.py")
mod = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(mod)

rows = pd.read_csv(fold_csv)
if "error" in rows.columns:
    rows = rows[rows["error"].isna() | (rows["error"].astype(str).str.strip() == "")].copy()

policies = [
    ("as_is_eval_threshold", None, None),
    ("val_balacc_rec>=0.88_spec>=0.20", 0.88, 0.20),
    ("val_balacc_rec>=0.88_spec>=0.25", 0.88, 0.25),
    ("val_balacc_rec>=0.90_spec>=0.20", 0.90, 0.20),
    ("val_balacc_rec>=0.90_spec>=0.25", 0.90, 0.25),
    ("val_balacc_rec>=0.92_spec>=0.20", 0.92, 0.20),
    ("val_balacc_rec>=0.92_spec>=0.25", 0.92, 0.25),
    ("val_balacc_rec>=0.92_spec>=0.30", 0.92, 0.30),
    ("val_balacc_rec>=0.95_spec>=0.25", 0.95, 0.25),
    ("val_balacc_rec>=0.95_spec>=0.30", 0.95, 0.30),
    ("val_balacc_rec>=0.95_spec>=0.35", 0.95, 0.35),
]

def pick_threshold(y, probs, rec_min, spec_min):
    grid = np.linspace(0.0, 1.0, 1001, dtype=float)
    best = None
    fallback = None
    for t in grid:
        pred = (probs >= t).astype(int)
        tp = int(((pred == 1) & (y == 1)).sum())
        tn = int(((pred == 0) & (y == 0)).sum())
        fp = int(((pred == 1) & (y == 0)).sum())
        fn = int(((pred == 0) & (y == 1)).sum())
        rec = tp / (tp + fn + 1e-12)
        spec = tn / (tn + fp + 1e-12)
        bal = 0.5 * (rec + spec)
        if rec >= rec_min and spec >= spec_min:
            key = (bal, rec + spec, -abs(t - 0.5))
            if best is None or key > best[0]:
                best = (key, float(t), float(rec), float(spec))
        else:
            shortfall = max(0.0, rec_min - rec) + max(0.0, spec_min - spec)
            key = (-shortfall, bal, rec + spec, -abs(t - 0.5))
            if fallback is None or key > fallback[0]:
                fallback = (key, float(t), float(rec), float(spec))
    if best is not None:
        _, thr, rec, spec = best
        return thr, rec, spec, True
    _, thr, rec, spec = fallback
    return thr, rec, spec, False

fold_rows = []
summary_rows = []
for policy, rec_min, spec_min in policies:
    per = []
    for _, r in rows.iterrows():
        val_df, val_y, val_domains, val_probs, _ = mod._predict_probs(Path(r["model_dir"]), Path(r["val_csv"]), str(r.get("feature_profile", "extended")))
        test_df, test_y, test_domains, test_probs, _ = mod._predict_probs(Path(r["model_dir"]), Path(r["test_csv"]), str(r.get("feature_profile", "extended")))
        if rec_min is None:
            thr = float(r.get("eval_threshold", r.get("report_threshold", 0.5)))
            met = True
        else:
            thr, _, _, met = pick_threshold(val_y.astype(int), val_probs.astype(float), rec_min, spec_min)
        m = mod._metrics_from_probs(test_df, test_y, test_domains, test_probs, threshold=thr)
        row = {"policy": policy, "fold": r["fold"], "threshold": thr, "val_constraints_met": bool(met)}
        for k in ["tp","tn","fp","fn","acc","prec","rec","f1","bal_acc","spec","auc","f1_real","f1_fake","auc_manip_audio_only","auc_manip_video_only","auc_manip_both_fake"]:
            row[k if not k.startswith("auc_manip") else k.replace("auc_manip_", "auc_")] = float(m.get(k, np.nan))
        per.append(row)
        fold_rows.append(row)

    p = pd.DataFrame(per)
    summary_rows.append({
        "policy": policy,
        "folds": int(len(p)),
        "constraints_met_folds": int(p["val_constraints_met"].sum()),
        "acc": float(p["acc"].mean()),
        "prec": float(p["prec"].mean()),
        "rec": float(p["rec"].mean()),
        "f1": float(p["f1"].mean()),
        "bal_acc": float(p["bal_acc"].mean()),
        "spec": float(p["spec"].mean()),
        "auc": float(p["auc"].mean()),
        "f1_real": float(p["f1_real"].mean()),
        "f1_fake": float(p["f1_fake"].mean()),
        "auc_audio_only": float(p["auc_audio_only"].mean()),
        "auc_video_only": float(p["auc_video_only"].mean()),
        "auc_both_fake": float(p["auc_both_fake"].mean()),
        "tp": float(p["tp"].sum()),
        "tn": float(p["tn"].sum()),
        "fp": float(p["fp"].sum()),
        "fn": float(p["fn"].sum()),
    })

summary = pd.DataFrame(summary_rows).sort_values(["bal_acc","acc","spec"], ascending=[False,False,False]).reset_index(drop=True)
fold_df = pd.DataFrame(fold_rows).sort_values(["policy","fold"]).reset_index(drop=True)

out_summary = logs / "comparison_nextsteps_20260314_step34_threshold_sweep.csv"
out_folds = logs / "comparison_nextsteps_20260314_step34_threshold_sweep_folds.csv"
summary.to_csv(out_summary, index=False)
fold_df.to_csv(out_folds, index=False)
print("Saved:", out_summary)
print("Saved:", out_folds)
display(summary.head(10))


In [ ]:
# Option-2: Step-35 retraining command (set RUN_STEP35=True to execute).
from pathlib import Path
import subprocess, shlex, sys

if "BACKEND_ROOT" in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if "BACKEND_ROOT" not in globals() or not (BACKEND_ROOT / "scripts" / "run_fakeav_mrdf_5fold_cv.py").exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / "backend"]
    BACKEND_ROOT = next((p for p in cands if (p / "scripts" / "run_fakeav_mrdf_5fold_cv.py").exists()), cands[-1])

PYTHON = BACKEND_ROOT / ".venv" / "bin" / "python"
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

RUN_STEP35 = False
cmd = [
    str(PYTHON), "scripts/run_fakeav_mrdf_5fold_cv.py",
    "--processed-csv", "data/processed/causal_multimodal_dataset_effnet_w2v2_physfix_fullvis_proxy.csv",
    "--out-root", "data/processed/causal_multimodal_dataset_fakeav_mrdf5cv",
    "--models-dir", "models",
    "--logs-dir", "models/experiment_logs",
    "--run-prefix", "fakeav_mrdf5cv_step35_realguard",
    "--split-mode", "full",
    "--n-splits", "5",
    "--feature-profile", "extended",
    "--loss", "bce",
    "--train-weight-application", "both",
    "--eval-threshold-source", "val_target",
    "--eval-threshold-priority", "balanced_acc",
    "--train-target-priority", "balanced_acc",
    "--target-acc", "0.90",
    "--target-precision", "0.87",
    "--target-recall", "0.88",
    "--target-f1", "0.88",
    "--target-spec", "0.4500",
    "--min-domain-spec", "0.3000",
    "--min-domain-rec", "0.8200",
    "--phase2-enable",
    "--phase2-rounds", "4",
    "--phase2-hardneg-source", "val",
    "--hard-negative-weight", "10.0",
    "--phase2-use-hard-positives",
    "--phase2-hardpos-source", "val",
    "--hard-positive-weight", "5.0",
    "--hardpos-scenario", "none",
    "--phase2-scenario-focus", "none",
    "--phase2-scenario-focus-weight", "1.0",
]

print("BACKEND_ROOT:", BACKEND_ROOT)
print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

if RUN_STEP35:
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f"run_fakeav_mrdf_5fold_cv.py failed: code={res.returncode}")
else:
    print("RUN_STEP35=False (skip)")


In [ ]:
# Compare Step-34 (as-is), Step-35, and best Step-34 threshold policy.
from pathlib import Path
import pandas as pd

if "BACKEND_ROOT" in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if "BACKEND_ROOT" not in globals() or not (BACKEND_ROOT / "models").exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / "backend"]
    BACKEND_ROOT = next((p for p in cands if (p / "models").exists()), cands[-1])

logs = BACKEND_ROOT / "models" / "experiment_logs"
run34 = logs / "fakeav_mrdf5cv_step34_fullvis_proxy_s42_20260314_161749" / "fold_metrics.csv"
run35 = logs / "fakeav_mrdf5cv_step35_realguard_s42_20260314_163548" / "fold_metrics.csv"
sweep34 = logs / "comparison_nextsteps_20260314_step34_threshold_sweep.csv"

def summarize(path, label):
    d = pd.read_csv(path)
    if "error" in d.columns:
        d = d[d["error"].isna() | (d["error"].astype(str).str.strip() == "")].copy()
    tn, fp, fn, tp = [int(d[c].sum()) for c in ["test_tn", "test_fp", "test_fn", "test_tp"]]
    return {
        "label": label,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "spec_real_recall": tn / (tn + fp + 1e-12),
        "rec_fake": tp / (tp + fn + 1e-12),
        "acc": float(d["test_acc"].mean()),
        "prec": float(d["test_prec"].mean()),
        "rec": float(d["test_rec"].mean()),
        "f1": float(d["test_f1"].mean()),
        "bal_acc": float(d["test_bal_acc"].mean()),
        "auc": float(d["test_auc"].mean()),
        "f1_real": float(d["test_f1_real"].mean()),
    }

rows = [summarize(run34, "step34_fullvis_proxy_as_is")]
if run35.exists():
    rows.append(summarize(run35, "step35_realguard_bce"))
if sweep34.exists():
    sw = pd.read_csv(sweep34)
    cand = sw[sw["acc"] >= 0.85].sort_values(["bal_acc", "spec", "acc"], ascending=[False, False, False]).head(1)
    if not cand.empty:
        r = cand.iloc[0]
        rows.append({
            "label": f"step34_policy:{r['policy']}",
            "tn": int(round(float(r["tn"]))), "fp": int(round(float(r["fp"]))),
            "fn": int(round(float(r["fn"]))), "tp": int(round(float(r["tp"]))),
            "spec_real_recall": float(r["spec"]), "rec_fake": float(r["rec"]),
            "acc": float(r["acc"]), "prec": float(r["prec"]), "rec": float(r["rec"]),
            "f1": float(r["f1"]), "bal_acc": float(r["bal_acc"]), "auc": float(r["auc"]),
            "f1_real": float(r["f1_real"]),
        })

out = pd.DataFrame(rows)
display(out)
out_csv = logs / "comparison_step34_step35_and_policy.csv"
out.to_csv(out_csv, index=False)
print("Saved:", out_csv)


## 36) Step-36 Multi-Seed Real-Guard + Ensemble

Goal: recover accuracy while preserving improved balanced-accuracy/specificity from Step-35.

Configuration used:
- Same protocol as Step-35 (`loss=bce`, `weight_application=both`, `target-spec=0.45`, domain guards, phase-2 hard examples).
- `seed-list=42,1337,2026`, `split-seed=42`.
- Ensemble enabled: `top_k=2`, ranking metric `val_bal_acc`, weighting `rank_metric`.


In [ ]:
# Step-36 execution cell (set RUN_STEP36_MULTI=True to run).
from pathlib import Path
import subprocess, shlex, sys

if "BACKEND_ROOT" in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if "BACKEND_ROOT" not in globals() or not (BACKEND_ROOT / "scripts" / "run_fakeav_mrdf_5fold_cv.py").exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / "backend"]
    BACKEND_ROOT = next((p for p in cands if (p / "scripts" / "run_fakeav_mrdf_5fold_cv.py").exists()), cands[-1])

PYTHON = BACKEND_ROOT / ".venv" / "bin" / "python"
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

RUN_STEP36_MULTI = False
cmd = [
    str(PYTHON), "scripts/run_fakeav_mrdf_5fold_cv.py",
    "--processed-csv", "data/processed/causal_multimodal_dataset_effnet_w2v2_physfix_fullvis_proxy.csv",
    "--out-root", "data/processed/causal_multimodal_dataset_fakeav_mrdf5cv",
    "--models-dir", "models",
    "--logs-dir", "models/experiment_logs",
    "--run-prefix", "fakeav_mrdf5cv_step36_realguard_multiseed",
    "--split-mode", "full",
    "--n-splits", "5",
    "--feature-profile", "extended",
    "--seed-list", "42,1337,2026",
    "--split-seed", "42",
    "--loss", "bce",
    "--train-weight-application", "both",
    "--eval-threshold-source", "val_target",
    "--eval-threshold-priority", "balanced_acc",
    "--train-target-priority", "balanced_acc",
    "--target-acc", "0.90",
    "--target-precision", "0.87",
    "--target-recall", "0.88",
    "--target-f1", "0.88",
    "--target-spec", "0.4500",
    "--min-domain-spec", "0.3000",
    "--min-domain-rec", "0.8200",
    "--phase2-enable",
    "--phase2-rounds", "4",
    "--phase2-hardneg-source", "val",
    "--hard-negative-weight", "10.0",
    "--phase2-use-hard-positives",
    "--phase2-hardpos-source", "val",
    "--hard-positive-weight", "5.0",
    "--hardpos-scenario", "none",
    "--phase2-scenario-focus", "none",
    "--phase2-scenario-focus-weight", "1.0",
    "--ensemble-enable",
    "--ensemble-top-k", "2",
    "--ensemble-rank-metric", "val_bal_acc",
    "--ensemble-weighting", "rank_metric",
]

print("BACKEND_ROOT:", BACKEND_ROOT)
print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

if RUN_STEP36_MULTI:
    res = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f"run_fakeav_mrdf_5fold_cv.py failed: code={res.returncode}")
else:
    print("RUN_STEP36_MULTI=False (skip)")


In [ ]:
# Step-36 summary table (includes Step-34/35 baselines and Step-36 seeds+ensemble).
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

if "BACKEND_ROOT" in globals():
    BACKEND_ROOT = Path(BACKEND_ROOT).expanduser().resolve()
if "BACKEND_ROOT" not in globals() or not (BACKEND_ROOT / "models").exists():
    cands = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / "backend"]
    BACKEND_ROOT = next((p for p in cands if (p / "models").exists()), cands[-1])

csv_path = BACKEND_ROOT / "models/experiment_logs/comparison_step34_step35_step36_multiseed.csv"
if not csv_path.exists():
    print("Missing:", csv_path)
else:
    d = pd.read_csv(csv_path)
    display(d)
    best = d.iloc[0]
    print("Best by bal_acc:", best["label"])
    print("  acc=", float(best["acc"]), "bal_acc=", float(best["bal_acc"]), "auc=", float(best["auc"]), "spec=", float(best["spec_real_recall"]))
